In [ ]:
# ─── 1. Dependencias e Importacoes (Colab / Local) ────────────────────────────
#!pip install -q ruptures scikit-learn openpyxl matplotlib pandas numpy scipy pyarrow joblib

import os, warnings, time, zipfile
from datetime import datetime
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.preprocessing import StandardScaler
from scipy import stats
from joblib import Parallel, delayed
import ruptures as rpt

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

%matplotlib inline
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#B0B0B0'
plt.rcParams['axes.linewidth'] = 0.8

TEMPOS = {}
print(f"OK deps | IN_COLAB={IN_COLAB} | CPUs={os.cpu_count()}")

In [ ]:
# ─── 2. Configuracao de Hiperparametros ───────────────────────────────────────
INPUT_FILE = 'sre_seg_200m_2110_2026_analitica.xlsx'
SRE_REFERENCIA = '320EGO0130' # sempre incluido na calibracao funcional e nas figuras de auditoria

JUMP     = 1          # Hiperparametro fixo
MIN_SIZE = 3          # n. minimo de intervalos por segmento (3 x 200 m = 600 m, piso operacional)

# ── Funcional (inalterado da v6): PELT l2 sobre IGG + banda +-MARGEM_PEN + BIC ──
MODEL_FUNC    = 'l2'
ALPHA_KM_FUNC = 3.0 / 25.23   # substituido pela calibracao da Celula 5 se CALIBRAR_FUNC
MARGEM_PEN    = 0.5
N_CAND_PEN    = 9
CALIBRAR_FUNC   = True
N_MIN_CALIB     = 50      # SREs com >= N intervalos entram na calibracao funcional
N_SRES_CALIB    = 120     # amostra estratificada por comprimento (deterministica)
ALPHA_GRID_FUNC = np.round(np.arange(0.05, 0.40 + 1e-9, 0.0125), 4)
CRITERIO_ALPHA_FUNC = 'objetivo'   # 'objetivo' = minimo da soma de BIC | 'cotovelo' = joelho quebras/km x alpha

# ── Estrutural (novo, Biswas & Kuna 2022): PELT 'normal' sobre d0 bruto + CROPS + cotovelo ──
MODEL_ESTRUT    = 'normal'  # log-verossimilhanca Gaussiana: muda media E/OU variancia (Eqs. 7-10)
MODO_PENALIDADE = 'rede'    # 'rede' = um beta para toda a malha (cotovelo da curva da rede)
                            # 'sre'  = cotovelo de cada SRE (procedimento literal do artigo)
BETA_MIN         = None     # inicio da varredura de beta; None = ln(N mediano dos SREs com d0 e N >= N_MIN_CALIB)
                            # (no artigo: beta de 5 a 35.6 com N = 161 -> ln(161) = 5.1)
BETA_MAX_QUANTIL = 0.90     # (rede) fim da curva = quantil do beta que zera as quebras de cada SRE
BETA_FIXO        = None     # (rede) beta fixo no lugar do cotovelo (ex.: para o artigo); None = cotovelo
                            # O cotovelo depende da faixa varrida (BETA_MIN, BETA_MAX_QUANTIL); a referencia
                            # BIC 3 ln(N) (media + variancia + posicao por quebra) e impressa na Celula 5.
N_GRID_BETA      = 400      # pontos da curva beta x n. de quebras (o caminho CROPS e exato; a grade so desenha/acha o cotovelo)
CROPS_MAX_RUNS   = 400      # limite de execucoes do PELT por SRE no CROPS
PELT_EXATO       = True     # poda adiada: otimo global exato com min_size > 1 (a poda do ruptures pode perde-lo)
VAR_FLOOR_FRAC   = 0.10     # piso de variancia = (frac x desvio do d0 no SRE)^2 -> trechos de d0 repetido nao viram quebra
N_MIN_COTOVELO_SRE = 4      # (sre) minimo de solucoes no caminho CROPS para um cotovelo proprio; senao usa o beta da rede

FUNDIR_FUNC   = True   # funcional: funde secoes vizinhas sem diferenca de media de IGG (Welch)
FUNDIR_ESTRUT = False  # B&K nao fundem (o cotovelo ja escolhe o n. de quebras); True = Welch + Brown-Forsythe
FUNDIR_COMB   = True   # combinada: funde se nao ha diferenca em d0 (media e variancia) nem em IGG (media)
ALPHA_TESTE   = 0.05   # nivel de significancia dos testes de fusao e da auditoria

SUBSEGMENTAR_COMB = True   # combinada em 2 etapas: PELT normal (mesmo beta) dentro de cada base funcional
BUFFER_KM         = 0.4    # quebra funcional a menos de BUFFER_KM de uma estrutural e absorvida por ela
MIN_COMP_COMB_KM  = 0.6
CV_LIMITE         = 1.0

# ── Auditoria de residuos (B&K, secao "Test of Assumptions") ──
N_LAGS_ACF      = 20       # lags da ACF / Ljung-Box (limitado a n/4)
N_MIN_AUDITORIA = 8        # SREs com menos d0 medidos nao sao auditados

GERAR_GRAFICOS = True
DPI            = 110
N_MIN_GRAFICO  = 5        # SREs com menos intervalos nao geram grafico
N_JOBS         = max(1, min(6, (os.cpu_count() or 2) - 2))  # cada worker ~250 MB; -1 = todos (cuidado com RAM)
USE_FAST_PELT  = True     # False = funcional usa rpt.Pelt (lento, mesmo resultado)
USAR_CACHE     = True     # cache parquet da leitura agregada

# ── Analise de sensibilidade da penalidade ──
SENS_ALVO    = 'estrutural'   # 'estrutural' = varia beta (PELT normal sobre d0) | 'funcional' = varia alpha_km (PELT l2 sobre IGG)
SENS_FATORES = [0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 2.00, 3.00]   # multiplicadores da penalidade calibrada (1.00 = referencia)
SENS_TOL_KM  = BUFFER_KM      # quebra a menos de SENS_TOL_KM de uma quebra da referencia conta como coincidente
SRES_ANALISE = [SRE_REFERENCIA]   # SREs analisados (rapido) | None = TODA A MALHA (~1.300 SREs com d0, ~20 mil graficos: ~1h a 1h30)
                                  # a calibracao (Celula 5) sempre usa a malha inteira, para o beta de referencia ser o da v8
SENS_GRADE      = 'crops'   # 'crops'   = um grafico por solucao distinta do caminho CROPS do SRE: penalidade crescente
                            #             e cada passo muda a segmentacao estrutural (so com SENS_ALVO='estrutural')
                            # 'fatores' = penalidade de referencia x SENS_FATORES (mesma grade para todos os SREs)
SENS_MAX_PASSOS = 30        # limite de graficos por SRE na grade 'crops' (amostra uniforme, sempre com a referencia)

TS = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_FILE = f'sensibilidade_rede_v8_{TS}.xlsx'
OUTPUT_DIR  = f'graficos_sensibilidade_v8_{TS}'
ZIP_FILE    = f'pacote_sensibilidade_v8_{TS}.zip'
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert MODO_PENALIDADE in ('rede', 'sre')
assert SENS_ALVO in ('estrutural', 'funcional') and SENS_GRADE in ('crops', 'fatores')
print(f"Configuracao carregada. Sensibilidade de {SENS_ALVO}: fatores {SENS_FATORES} | Min_size={MIN_SIZE} | Jump={JUMP} | estrutural={MODEL_ESTRUT} sobre d0 | "
      f"penalidade={MODO_PENALIDADE} | piso_var={VAR_FLOOR_FRAC}")

In [ ]:
# ─── 3. Leitura de Dados (com cache) e Calculo do ACA ─────────────────────────
t0 = time.time()
if not os.path.exists(INPUT_FILE) and IN_COLAB:
    print(f"Arquivo '{INPUT_FILE}' nao encontrado. Selecione o arquivo para upload:")
    uploaded = files.upload()
    if uploaded:
        INPUT_FILE = list(uploaded.keys())[0]

COLS_LEITURA = ['sre', 'sre_segmento', 'km_inicial', 'km_final', 'igg', 'iri_medio', 'd0']

def carregar_dados(path):
    df = pd.read_excel(path, usecols=lambda c: c in COLS_LEITURA)
    for c in ['km_inicial', 'km_final', 'igg', 'iri_medio', 'd0']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    has_seg = 'sre_segmento' in df.columns
    group_cols = ['sre', 'sre_segmento'] if has_seg else ['sre', 'km_inicial', 'km_final']
    agg = df.groupby(group_cols, as_index=False).agg(
        km_inicial=('km_inicial', 'min'), km_final=('km_final', 'max'),
        igg=('igg', 'mean'), iri=('iri_medio', 'mean'), d0=('d0', 'mean'))
    agg = agg.dropna(subset=['km_inicial', 'km_final']).sort_values(['sre', 'km_inicial']).reset_index(drop=True)

    agg['d0_obs'] = agg['d0'].notna()   # d0 medido (antes do preenchimento) -> usado na auditoria
    for c in ['igg', 'iri', 'd0']:
        agg[c] = agg.groupby('sre')[c].transform(lambda s: s.interpolate(limit_direction='both').fillna(s.median()))

    agg['km_centro_km'] = 0.5 * (agg['km_inicial'] + agg['km_final'])
    agg['aca'] = agg.groupby('sre')['d0'].transform(lambda s: (s - s.mean()).cumsum())
    agg['sre'] = agg['sre'].astype(str)
    return agg

def carregar_dados_cache(path):
    cache = os.path.splitext(path)[0] + '_agg_cache_v7.parquet'
    if USAR_CACHE and os.path.exists(cache) and os.path.getmtime(cache) >= os.path.getmtime(path):
        print(f"Lendo cache: {cache}")
        return pd.read_parquet(cache)
    agg = carregar_dados(path)
    if USAR_CACHE:
        agg.to_parquet(cache, index=False)
        print(f"Cache gravado: {cache}")
    return agg

df = carregar_dados_cache(INPUT_FILE)
sres = sorted(df['sre'].unique().tolist())

# arrays numpy por SRE (evita filtrar o DataFrame inteiro a cada SRE)
grupos = {
    sre: {
        'ini': g['km_inicial'].to_numpy(float), 'fim': g['km_final'].to_numpy(float),
        'centro': g['km_centro_km'].to_numpy(float), 'igg': g['igg'].to_numpy(float),
        'iri': g['iri'].to_numpy(float), 'd0': g['d0'].to_numpy(float), 'aca': g['aca'].to_numpy(float),
        'd0_obs': g['d0_obs'].to_numpy(bool),
    }
    for sre, g in df.groupby('sre', sort=True)
}
TEMPOS['leitura'] = time.time() - t0
sem_d0  = sum(not np.isfinite(a['d0']).any() for a in grupos.values())
sem_igg = sum(not np.isfinite(a['igg']).any() for a in grupos.values())
com_d0  = [a for a in grupos.values() if np.isfinite(a['d0']).all() and len(a['d0']) > 1]
rep_d0  = np.mean(np.concatenate([np.diff(a['d0']) == 0 for a in com_d0])) if com_d0 else 0.0
print(f"SREs: {len(sres)} | Total intervalos: {len(df)} | d0 medido: {df['d0_obs'].mean():.1%} | {TEMPOS['leitura']:.1f}s")
print(f"AVISO: {sem_d0} SREs sem nenhum d0 e {sem_igg} sem nenhum IGG (saem com 1 segmento e estatisticas vazias)")
print(f"d0 consecutivos identicos (preenchimento de lacunas): {rep_d0:.1%} -> tratados pelo piso de variancia")

In [ ]:
# ─── 4. Funcoes Auxiliares (PELT rapido normal/l2, CROPS, cotovelo, fusao, auditoria) ─
# ── PELT rapido: replica rpt.Pelt(model, min_size, jump).fit(x).predict(pen) p/ sinal 1-D ──
# 'normal' = CostNormal do ruptures: custo do segmento = n * log(var), var com ddof=0, ou seja
# -2 x log-verossimilhanca Gaussiana com media E variancia proprias (Biswas & Kuna, 2022, Eqs. 7-10).
# O ruptures soma 1e-6 a var (piso dependente da unidade); aqui o piso e var_floor, relativo ao SRE.
# Um piso aditivo preserva a desigualdade C(a,b) + C(b,c) <= C(a,c), logo a poda do PELT continua exata.
_MIN_SIZE_CUSTO = {'l2': 1, 'normal': 2}  # cost.min_size interno do ruptures 1.1.x

def prefixos_pelt(y):
    y = np.asarray(y, float).ravel()
    y = y - y.mean()                 # custos l2/normal invariantes a translacao; reduz cancelamento numerico
    z = np.zeros(1)
    return np.concatenate([z, np.cumsum(y)]), np.concatenate([z, np.cumsum(y * y)])

def custos_segmentos(model, ts, e, P, var_floor=0.0):
    """Custo do segmento [t, e) para um vetor de inicios ts (mesmas formulas de CostL2 / CostNormal)."""
    S1, S2 = P
    n = (e - ts).astype(float)
    s1 = S1[e] - S1[ts]
    sq = S2[e] - S2[ts]
    if model == 'l2':
        return sq - s1 * s1 / n
    var = np.maximum(sq / n - (s1 / n) ** 2, 0.0)
    return n * np.log(var + var_floor)

def pelt_fast(signal, model, min_size, jump, pen, P=None, var_floor=0.0, exato=False):
    """PELT com custo O(1) por segmento; retorna bkps no formato do ruptures (inclui n).
    exato=False: poda identica ao rpt.Pelt (com min_size > 1 ela pode descartar o otimo em casos raros).
    exato=True : poda adiada -- t podado em s so sai a partir de s + min_size, quando (s, T) ja e um
                 ultimo segmento admissivel; recupera o otimo global exato (Killick et al., 2012)."""
    y = np.asarray(signal, float).ravel()
    n = len(y)
    min_size = max(int(min_size), _MIN_SIZE_CUSTO[model])
    if P is None:
        P = prefixos_pelt(y)
    F = np.full(n + 1, np.inf); F[0] = 0.0
    prev = np.zeros(n + 1, dtype=int)
    valido = np.zeros(n + 1, dtype=bool); valido[0] = True
    sai_em = np.full(n + 1, n + 1)               # (exato) instante a partir do qual o candidato e podado
    adm = np.empty(0, dtype=int)
    ind = [k for k in range(0, n, jump) if k >= min_size] + [n]
    for bkp in ind:
        novo = (bkp - min_size) // jump * jump
        if novo >= 0:
            adm = np.append(adm, novo)
        ts = adm[valido[adm] & (sai_em[adm] > bkp)]
        if len(ts) == 0:
            raise ValueError('sem particao admissivel')
        tot = F[ts] + (custos_segmentos(model, ts, bkp, P, var_floor) + pen)
        j = int(np.argmin(tot))
        F[bkp] = tot[j]; prev[bkp] = ts[j]; valido[bkp] = True
        if exato:
            podar = ts[tot > F[bkp] + pen]
            sai_em[podar] = np.minimum(sai_em[podar], bkp + min_size)
            adm = ts
        else:
            adm = ts[tot <= F[bkp] + pen]
    bkps, e = [], n
    while e > 0:
        bkps.append(int(e)); e = prev[e]
    return bkps[::-1]

def custo_total(model, bkps, P, var_floor=0.0):
    """Custo nao penalizado Q da segmentacao (soma dos custos dos segmentos)."""
    ini = np.array([0] + list(bkps[:-1])); fim = np.array(bkps)
    return float(sum(custos_segmentos(model, ini[i:i + 1], fim[i], P, var_floor)[0] for i in range(len(fim))))

def piso_variancia(d0):
    s = float(np.nanstd(d0)) if np.isfinite(d0).any() else 0.0
    return max((VAR_FLOOR_FRAC * s) ** 2, 1e-9)

# ── CROPS (Haynes, Eckley & Fearnhead, 2017): caminho exato de penalidades ──
def crops_normal(y, beta_min, min_size=MIN_SIZE, jump=JUMP, var_floor=1e-9, max_runs=CROPS_MAX_RUNS):
    """Todas as segmentacoes otimas do PELT 'normal' para beta em [beta_min, inf).
    Retorna {K: (Q, bkps)} so com os vertices da envoltoria convexa inferior de (K, Q), que sao
    exatamente as solucoes otimas para algum beta: K* (beta) = argmin_K Q_K + beta * K."""
    y = np.asarray(y, float).ravel()
    N = len(y)
    P = prefixos_pelt(y)
    sols = {0: (custo_total('normal', [N], P, var_floor), [N])}   # 0 quebras e sempre admissivel
    if N < 2 * max(min_size, _MIN_SIZE_CUSTO['normal']):
        return sols

    def run(beta):
        bk = pelt_fast(y, 'normal', min_size, jump, float(beta), P, var_floor, exato=PELT_EXATO)
        K, Q = len(bk) - 1, custo_total('normal', bk, P, var_floor)
        if K not in sols or Q < sols[K][0] - 1e-9:
            sols[K] = (Q, bk)
        return K, sols[K][0]

    K_lo, Q_lo = run(beta_min)
    b_hi = max(2.0 * beta_min, 1.0)
    K_hi, Q_hi = run(b_hi)
    n_runs = 2
    while K_hi > 0 and n_runs < max_runs:
        b_hi *= 2.0
        K_hi, Q_hi = run(b_hi); n_runs += 1
    pilha = [(beta_min, K_lo, Q_lo, b_hi, K_hi, Q_hi)]
    while pilha and n_runs < max_runs:
        b0, K0, Q0, b1, K1, Q1 = pilha.pop()
        if K0 <= K1 + 1:
            continue
        bi = (Q1 - Q0) / (K0 - K1)          # beta em que as duas solucoes empatam
        if not (b0 < bi < b1):
            continue
        Ki, Qi = run(bi); n_runs += 1
        if Ki not in (K0, K1):              # existe solucao intermediaria -> subdivide
            pilha += [(b0, K0, Q0, bi, Ki, Qi), (bi, Ki, Qi, b1, K1, Q1)]

    hull = []                               # envoltoria convexa inferior (monotone chain)
    for K in sorted(sols):
        while len(hull) >= 2:
            (k1, q1), (k2, q2) = hull[-2], hull[-1]
            if (k2 - k1) * (sols[K][0] - q1) - (q2 - q1) * (K - k1) <= 0:
                hull.pop()
            else:
                break
        hull.append((K, sols[K][0]))
    return {K: sols[K] for K, _ in hull}

def tabela_caminho(path, beta_min):
    """K, Q e intervalo [beta_inf, beta_sup] em que cada segmentacao do caminho e otima."""
    Ks = np.array(sorted(path), dtype=int)
    Q = np.array([path[k][0] for k in Ks])
    sup = np.r_[np.inf, (Q[:-1] - Q[1:]) / (Ks[1:] - Ks[:-1])]
    inf_ = np.r_[sup[1:], beta_min]
    return Ks, Q, inf_, sup

def k_otimo(Ks, Q, betas):
    betas = np.atleast_1d(np.asarray(betas, float))
    return Ks[np.argmin(Q[None, :] + betas[:, None] * Ks[None, :], axis=1)]

def kneedle_idx(x, y):
    """Cotovelo da curva (maior distancia a corda entre extremos, eixos normalizados)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x) < 3 or np.ptp(y) == 0:
        return 0
    xn = (x - x.min()) / max(np.ptp(x), 1e-12)
    yn = (y - y.min()) / max(np.ptp(y), 1e-12)
    dist = np.abs((yn[-1] - yn[0]) * xn - (xn[-1] - xn[0]) * yn + xn[-1] * yn[0] - yn[-1] * xn[0])
    return int(np.argmax(dist))

def kneedle(x, y):
    return float(np.asarray(x, float)[kneedle_idx(x, y)])

def cotovelo_sre(Ks, Q, inf_, sup, beta_min, n_grid=N_GRID_BETA):
    """Grafico de diagnostico do SRE (B&K, Figs. 8-9): n. de quebras x beta de beta_min ate o beta que
    zera as quebras; o cotovelo define K*, e o beta escolhido e o centro do intervalo em que K* e otimo."""
    if len(Ks) < N_MIN_COTOVELO_SRE:
        return None
    grid = np.linspace(beta_min, sup[1], n_grid)
    K_est = int(k_otimo(Ks, Q, [kneedle(grid, k_otimo(Ks, Q, grid))])[0])
    i = int(np.flatnonzero(Ks == K_est)[0])
    beta_sel = 0.5 * (inf_[i] + sup[i]) if np.isfinite(sup[i]) else sup[1] * 1.05
    return K_est, float(beta_sel)

# ── Funcional (igual a v6) ──
def banda_penalidades(base_pen, margem=MARGEM_PEN, n=N_CAND_PEN):
    base_pen = float(base_pen)
    if base_pen <= 0 or margem <= 0 or n <= 1:
        return [max(base_pen, 0.05)]
    lo = max(0.05, base_pen * (1.0 - margem))
    hi = base_pen * (1.0 + margem)
    return [float(v) for v in np.linspace(lo, hi, int(n))]

def pelt_breaks_multipen(Xs, modelo, min_size, jump, pens):
    Xs = np.asarray(Xs, float)
    if USE_FAST_PELT and modelo in _MIN_SIZE_CUSTO and Xs.reshape(len(Xs), -1).shape[1] == 1:
        y = Xs.ravel()
        P = prefixos_pelt(y)
        out = {}
        for pen in pens:
            try:
                bkps = pelt_fast(y, modelo, min_size, jump, float(pen), P)
                out[float(pen)] = [int(b) - 1 for b in bkps[:-1]]
            except Exception:
                out[float(pen)] = []
        return out
    try:
        algo = rpt.Pelt(model=modelo, min_size=min_size, jump=jump).fit(Xs)
    except Exception:
        return {float(p): [] for p in pens}
    out = {}
    for pen in pens:
        try:
            bkps = algo.predict(pen=float(pen))
            out[float(pen)] = [int(b) - 1 for b in bkps[:-1]]
        except Exception:
            out[float(pen)] = []
    return out

def labels_from_bkps(bkps, N):
    labels = np.zeros(N, dtype=int)
    prev = 0
    for s_idx, bp in enumerate(bkps):
        labels[prev:int(bp)] = s_idx
        prev = int(bp)
    return labels

def bic_l2(X, labels):
    N = len(X)
    D = X.shape[1]
    sse = 0.0
    for lbl in np.unique(labels):
        pts = X[labels == lbl]
        sse += float(((pts - pts.mean(axis=0)) ** 2).sum())
    k = len(np.unique(labels)) * D + (len(np.unique(labels)) - 1)
    return N * np.log(max(sse, 1e-12) / N) + k * np.log(N)

def media_cv(v):
    if len(v) == 0:
        return None, None
    m = float(np.mean(v))
    s = float(np.std(v)) if len(v) > 1 else 0.0
    cv = s / abs(m) if abs(m) > 1e-9 else np.nan
    return m, cv

# ── Fusao de secoes vizinhas (CDA modificado, B&K Fig. 3) ──
def _p_welch(x, y):
    """Diferenca de media (Welch)."""
    if len(x) < 2 or len(y) < 2:
        return 1.0                      # amostra pequena demais para provar diferenca
    with warnings.catch_warnings():     # workers do joblib nao herdam o filtro da Celula 1
        warnings.simplefilter('ignore')
        p = stats.ttest_ind(x, y, equal_var=False).pvalue
    if np.isnan(p):                     # variancias nulas
        return 1.0 if np.isclose(np.mean(x), np.mean(y)) else 0.0
    return float(p)

def _p_bf(x, y):
    """Diferenca de variancia (Brown-Forsythe = Levene com mediana; robusto a nao-normalidade)."""
    if len(x) < 2 or len(y) < 2:
        return 1.0
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        p = stats.levene(x, y, center='median').pvalue
    if np.isnan(p):
        return 1.0 if np.isclose(np.var(x), np.var(y)) else 0.0
    return float(p)

def fundir_testes(inicios, series, N, alpha=ALPHA_TESTE):
    """Funde secoes consecutivas sem diferenca significativa em NENHUMA das series.
    `series` = [(array, testa_variancia)]; com testa_variancia=True a quebra so cai se nem a media (Welch)
    nem a variancia (Brown-Forsythe) diferem, preservando as quebras de variancia do custo 'normal'.
    `inicios` = indices do primeiro intervalo de cada segmento (exceto o 0). Funde primeiro o par mais parecido."""
    fr = sorted(int(s) for s in inicios)
    while fr:
        lim = [0] + fr + [N]
        ps = []
        for i in range(len(fr)):
            p = 1.0
            for s, testa_var in series:
                x, y = s[lim[i]:lim[i + 1]], s[lim[i + 1]:lim[i + 2]]
                p = min(p, _p_welch(x, y), _p_bf(x, y) if testa_var else 1.0)
            ps.append(p)
        j = int(np.argmax(ps))
        if ps[j] <= alpha:
            break
        fr.pop(j)
    return fr

# ── Segmentacoes (indices: cada quebra = indice do 1o intervalo do segmento seguinte) ──
def _extensao(a, lo, hi):
    fim = a['ini'][hi] if hi < len(a['ini']) else float(np.max(a['fim'][lo:hi]))
    return float(fim - np.min(a['ini'][lo:hi]))

def segmentar_funcional_sre(a, alpha):
    igg = a['igg']
    N = len(igg)
    pens = banda_penalidades(alpha * _extensao(a, 0, N), MARGEM_PEN, N_CAND_PEN)
    X = np.asarray(igg, float).reshape(-1, 1)
    Xs = StandardScaler().fit_transform(X) if N > 1 else X
    breaks_map = pelt_breaks_multipen(Xs, MODEL_FUNC, MIN_SIZE, JUMP, pens)
    melhor = None
    for pen in pens:
        breaks = breaks_map[pen]
        labels = labels_from_bkps([b + 1 for b in breaks] + [N], N)
        bic = bic_l2(Xs, labels)
        if melhor is None or bic < melhor[0]:
            melhor = (bic, pen, breaks)
    bic, pen, breaks = melhor
    return [b + 1 for b in breaks], pen, bic

def segmentar_estrutural_trecho(d0, lo, hi, beta, var_floor):
    """PELT 'normal' sobre d0 bruto em [lo, hi) com penalidade beta (sem padronizar, sem snap)."""
    if hi - lo < 2 * MIN_SIZE:
        return []
    bk = pelt_fast(d0[lo:hi], MODEL_ESTRUT, MIN_SIZE, JUMP, float(beta), var_floor=var_floor, exato=PELT_EXATO)
    return [lo + b for b in bk[:-1]]

def reconciliar_combinada(func_km, sub_km_raw, km_min, km_max, min_comp_km=MIN_COMP_COMB_KM, buffer_km=BUFFER_KM):
    fronteiras = list(sub_km_raw)
    for fk in func_km:
        if not any(abs(fk - sk) <= buffer_km for sk in sub_km_raw):
            fronteiras.append(fk)
    fronteiras = sorted(set(fronteiras))
    limpas = []
    ultimo = km_min
    for f in fronteiras:
        if (f - ultimo) >= min_comp_km and (km_max - f) >= min_comp_km:
            limpas.append(f)
            ultimo = f
    return limpas

# ── Auditoria de residuos (B&K, "Test of Assumptions": Tab. 1, Figs. 11-13) ──
def residuos_segmentos(y, obs, inicios):
    """Residuo bruto r = d0 - media do segmento e padronizado z = r / desvio do segmento (ddof=1),
    so nos d0 medidos. O modelo 'normal' admite variancia propria por segmento, entao a normalidade
    e testada em z; r (como no artigo) mistura variancias e tende a caudas pesadas."""
    N = len(y)
    lim = [0] + sorted(int(s) for s in inicios) + [N]
    r = np.full(N, np.nan); z = np.full(N, np.nan); grupos_r, n_sd = [], []
    for i in range(len(lim) - 1):
        idx = np.arange(lim[i], lim[i + 1])[obs[lim[i]:lim[i + 1]]]
        if len(idx) == 0:
            continue
        v = y[idx]
        r[idx] = v - v.mean()
        if len(idx) > 1:
            sd = float(v.std(ddof=1))
            n_sd.append((len(idx), sd))
            grupos_r.append(r[idx])
            if sd > 0:
                z[idx] = r[idx] / sd
    return r, z, grupos_r, n_sd

def auditar_residuos(y, obs, inicios, n_lags=N_LAGS_ACF, alpha=ALPHA_TESTE):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return _auditar_residuos(y, obs, inicios, n_lags, alpha)

def _auditar_residuos(y, obs, inicios, n_lags, alpha):
    r, z, grupos_r, n_sd = residuos_segmentos(y, obs, inicios)
    rr, zz = r[np.isfinite(r)], z[np.isfinite(z)]
    out = {'n_residuos': int(len(rr)), 'n_segmentos': len(inicios) + 1}
    if len(rr) < N_MIN_AUDITORIA or len(zz) < 3 or np.ptp(zz) == 0:
        return out, z
    sw_W, sw_p = stats.shapiro(zz) if len(zz) <= 5000 else (np.nan, np.nan)
    ks_D, ks_p = stats.kstest(zz, 'norm')
    sw_p_bruto = stats.shapiro(rr).pvalue if 3 <= len(rr) <= 5000 and np.ptp(rr) > 0 else np.nan
    # ACF dos residuos ao longo do km (B&K Fig. 12) + Ljung-Box
    n = len(rr)
    h = int(max(1, min(n_lags, n // 4)))
    rc = rr - rr.mean(); den = float((rc ** 2).sum())
    acf = np.array([(rc[:-k] * rc[k:]).sum() / den for k in range(1, h + 1)]) if den > 0 else np.zeros(h)
    banda = 1.96 / np.sqrt(n)
    lb_Q = n * (n + 2) * float(np.sum(acf ** 2 / (n - np.arange(1, h + 1))))
    lb_p = float(stats.chi2.sf(lb_Q, h))
    # heterocedasticidade entre segmentos (B&K Fig. 13) -> justifica 'normal' em vez de 'l2'
    validos = [g for g in grupos_r if len(g) >= 2]
    bf_p = float(stats.levene(*validos, center='median').pvalue) if len(validos) >= 2 else np.nan
    sds = np.array([s for _, s in n_sd if s > 0])
    out.update({
        'shapiro_W': round(float(sw_W), 4), 'shapiro_p': round(float(sw_p), 4),
        'ks_D': round(float(ks_D), 4), 'ks_p': round(float(ks_p), 4),
        'shapiro_p_res_bruto': round(float(sw_p_bruto), 4),
        'normalidade_ok': bool(sw_p > alpha and ks_p > alpha),
        'acf_lag1': round(float(acf[0]), 4), 'acf_max_abs': round(float(np.max(np.abs(acf))), 4),
        'banda_acf_95': round(float(banda), 4), 'lags_fora_banda': int(np.sum(np.abs(acf) > banda)), 'n_lags': h,
        'ljung_box_Q': round(lb_Q, 3), 'ljung_box_p': round(lb_p, 4), 'independencia_ok': bool(lb_p > alpha),
        'brown_forsythe_p': round(bf_p, 4) if np.isfinite(bf_p) else np.nan,
        'heterocedastico': bool(np.isfinite(bf_p) and bf_p <= alpha),
        'razao_desvio_max_min': round(float(sds.max() / sds.min()), 3) if len(sds) >= 2 else np.nan,
        'erro_padrao_medio': round(float(np.mean([s / np.sqrt(k) for k, s in n_sd])), 3) if n_sd else np.nan,
    })
    return out, z

print("OK helpers")

In [ ]:
# ─── 5. Calibracao: alpha funcional (BIC) + caminhos CROPS e cotovelo estrutural ──
# ── A) Funcional: grade de alpha_km avaliada nos SREs de calibracao (igual a v6) ──
def _calib_func_sre(a, grid_f):
    N = len(a['ini'])
    ext = max(_extensao(a, 0, N), 1e-9)
    bic, nq = [], []
    for al in grid_f:
        fr, _, b = segmentar_funcional_sre(a, al)
        bic.append(b); nq.append(len(fr) / ext)
    return np.array(bic), np.array(nq)

def escolher_sres_calibracao(grupos):
    # so entram SREs com IGG medido (sem dados -> objetivo constante, so dilui a soma)
    tam = pd.Series({s: len(a['ini']) for s, a in grupos.items() if np.isfinite(a['igg']).all()})
    cand = tam[tam >= N_MIN_CALIB].sort_values(kind='stable')
    if cand.empty:
        cand = tam.sort_values(kind='stable').tail(10)
    if len(cand) > N_SRES_CALIB:   # amostra estratificada por comprimento (sempre a mesma)
        cand = cand.iloc[np.unique(np.linspace(0, len(cand) - 1, N_SRES_CALIB).round().astype(int))]
    sel = list(cand.index)
    if SRE_REFERENCIA in grupos and SRE_REFERENCIA not in sel:
        sel.append(SRE_REFERENCIA)
    return sel

def calibrar_funcional(grupos):
    sel = escolher_sres_calibracao(grupos)
    res = Parallel(n_jobs=N_JOBS)(delayed(_calib_func_sre)(grupos[s], ALPHA_GRID_FUNC) for s in sel)
    obj = np.nansum(np.vstack([r[0] for r in res]), axis=0)   # verossimilhanca conjunta (min)
    nq = np.vstack([r[1] for r in res]).mean(axis=0)
    a_obj = float(ALPHA_GRID_FUNC[int(np.argmin(obj))])
    a_cot = kneedle(ALPHA_GRID_FUNC, nq)
    if a_obj in (float(ALPHA_GRID_FUNC[0]), float(ALPHA_GRID_FUNC[-1])):
        print(f"  AVISO: otimo do BIC funcional na borda da grade ({a_obj:.4f}) -> prefira 'cotovelo'")
    best = a_cot if CRITERIO_ALPHA_FUNC == 'cotovelo' else a_obj
    curva = pd.DataFrame({'alpha_km': ALPHA_GRID_FUNC, 'soma_bic': obj, 'quebras_por_km': nq})
    print(f"  Funcional ({len(sel)} SREs): objetivo={a_obj:.4f} | cotovelo={a_cot:.4f} -> "
          f"ALPHA_KM_FUNC={best:.4f} ({CRITERIO_ALPHA_FUNC})")
    return best, curva, sel, (nq, obj, a_obj, a_cot)

# ── B) Estrutural: caminho CROPS exato de cada SRE (PELT 'normal' sobre d0 bruto) ──
def caminho_sre(a, beta_min):
    d0 = a['d0']
    path = crops_normal(d0, beta_min, MIN_SIZE, JUMP, piso_variancia(d0))
    Ks, Q, inf_, sup = tabela_caminho(path, beta_min)
    return {'Ks': Ks, 'Q': Q, 'inf': inf_, 'sup': sup, 'bkps': {int(k): path[k][1] for k in Ks}}

t0 = time.time()
if CALIBRAR_FUNC:
    ALPHA_KM_FUNC_OPT, CURVA_FUNC, SRES_CALIB, _cf = calibrar_funcional(grupos)
else:
    ALPHA_KM_FUNC_OPT, CURVA_FUNC, SRES_CALIB, _cf = ALPHA_KM_FUNC, None, [], None
TEMPOS['calib_funcional'] = time.time() - t0

t0 = time.time()
N_D0 = pd.Series({s: len(a['d0']) for s, a in grupos.items() if np.isfinite(a['d0']).all()}, dtype=float)
if BETA_MIN is None:
    base = N_D0[N_D0 >= N_MIN_CALIB] if (N_D0 >= N_MIN_CALIB).any() else N_D0
    BETA_MIN_EF = float(np.log(np.median(base)))
else:
    BETA_MIN_EF = float(BETA_MIN)
_base_bic = N_D0[N_D0 >= N_MIN_CALIB] if (N_D0 >= N_MIN_CALIB).any() else N_D0
BETA_BIC = float(3 * np.log(np.median(_base_bic)))   # referencia: BIC com 3 parametros por quebra (mu, sigma, posicao)
SRES_D0 = [s for s in sres if s in N_D0.index and N_D0[s] >= 2 * MIN_SIZE]
_cams = Parallel(n_jobs=N_JOBS, batch_size=8)(delayed(caminho_sre)(grupos[s], BETA_MIN_EF) for s in SRES_D0)
CAMINHOS = dict(zip(SRES_D0, _cams))
EXT = {s: _extensao(grupos[s], 0, len(grupos[s]['ini'])) for s in sres}

# ── C) Grafico de diagnostico da REDE: beta x total de quebras (B&K Fig. 9, somado nos SREs) ──
BETA_ZERO = pd.Series({s: c['sup'][1] for s, c in CAMINHOS.items() if len(c['Ks']) > 1})
BETA_MAX_REDE = float(BETA_ZERO.quantile(BETA_MAX_QUANTIL)) if len(BETA_ZERO) else 10 * BETA_MIN_EF
GRID_BETA = np.linspace(BETA_MIN_EF, BETA_MAX_REDE, N_GRID_BETA)
K_REDE = np.zeros(len(GRID_BETA))
for c in CAMINHOS.values():
    K_REDE += k_otimo(c['Ks'], c['Q'], GRID_BETA)
EXT_D0 = sum(EXT[s] for s in CAMINHOS)
BETA_COTOVELO_REDE = kneedle(GRID_BETA, K_REDE)
BETA_REDE = float(BETA_FIXO) if BETA_FIXO is not None else BETA_COTOVELO_REDE

# ── D) Cotovelo de cada SRE (procedimento literal do artigo) e beta usado ──
COTOVELO = {s: cotovelo_sre(c['Ks'], c['Q'], c['inf'], c['sup'], BETA_MIN_EF) for s, c in CAMINHOS.items()}
BETA_SRE, ORIGEM_BETA = {}, {}
for s in CAMINHOS:
    if MODO_PENALIDADE == 'sre' and COTOVELO[s] is not None:
        BETA_SRE[s], ORIGEM_BETA[s] = COTOVELO[s][1], 'cotovelo_sre'
    else:
        BETA_SRE[s], ORIGEM_BETA[s] = BETA_REDE, ('rede' if MODO_PENALIDADE == 'rede' else 'fallback_rede')
TEMPOS['calib_estrutural'] = time.time() - t0

DIAG_REDE = pd.DataFrame({'beta': GRID_BETA, 'quebras_rede': K_REDE.astype(int),
                          'quebras_por_km': K_REDE / max(EXT_D0, 1e-9)})
TAB_CROPS = pd.DataFrame([
    {'sre': s, 'K_quebras': int(k), 'custo_Q': round(float(q), 4), 'beta_inf': round(float(bi), 4),
     'beta_sup': (round(float(bs), 4) if np.isfinite(bs) else None),
     'cotovelo_sre': bool(COTOVELO[s] is not None and COTOVELO[s][0] == k),
     'escolhido': bool(k == k_otimo(c['Ks'], c['Q'], [BETA_SRE[s]])[0])}
    for s, c in CAMINHOS.items() for k, q, bi, bs in zip(c['Ks'], c['Q'], c['inf'], c['sup'])])

b_cot = np.array([v[1] for v in COTOVELO.values() if v is not None])
k_rede_sel = int(K_REDE[np.argmin(np.abs(GRID_BETA - BETA_REDE))])
print(f"  Estrutural: CROPS em {len(CAMINHOS)} SREs com d0 ({TAB_CROPS.shape[0]} solucoes otimas) | "
      f"beta de {BETA_MIN_EF:.2f} a {BETA_MAX_REDE:.2f} (q{int(BETA_MAX_QUANTIL * 100)} do beta que zera as quebras)")
print(f"  Cotovelo da rede: beta={BETA_COTOVELO_REDE:.2f} | referencia BIC 3 ln(N)={BETA_BIC:.2f}"
      + (f" | BETA_FIXO={BETA_FIXO}" if BETA_FIXO is not None else ''))
print(f"  BETA_REDE={BETA_REDE:.2f} -> {k_rede_sel} quebras na malha ({k_rede_sel / max(EXT_D0, 1e-9):.3f}/km)")
if len(b_cot):
    print(f"  Cotovelo por SRE ({len(b_cot)} SREs com >= {N_MIN_COTOVELO_SRE} solucoes): beta mediano={np.median(b_cot):.2f} "
          f"(p25={np.percentile(b_cot, 25):.2f}, p75={np.percentile(b_cot, 75):.2f})")
print(f"✓ Usando: ALPHA_KM_FUNC={ALPHA_KM_FUNC_OPT:.4f} | estrutural MODO_PENALIDADE='{MODO_PENALIDADE}'"
      + (f" (beta={BETA_REDE:.2f})" if MODO_PENALIDADE == 'rede' else
         f" ({sum(v == 'cotovelo_sre' for v in ORIGEM_BETA.values())} cotovelos proprios, "
         f"{sum(v == 'fallback_rede' for v in ORIGEM_BETA.values())} com beta da rede)"))

# ── Figura de calibracao ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax = axes[0, 0]
if _cf is not None:
    nq, obj, a_obj, a_cot = _cf
    ax.plot(ALPHA_GRID_FUNC, nq, 'o-', color='#1F4E79', ms=3, label='quebras/km (media)')
    ax2 = ax.twinx(); ax2.plot(ALPHA_GRID_FUNC, obj, color='#C65A57', lw=1.2); ax2.set_ylabel('soma BIC', color='#C65A57')
    ax.axvline(a_obj, color='#C65A57', lw=1.2, label=f'BIC min: {a_obj:.4f}')
    ax.axvline(a_cot, color='#555555', ls='--', lw=1.2, label=f'cotovelo: {a_cot:.4f}')
    ax.legend(fontsize=8, loc='upper right')
ax.set_xlabel('alpha_km (penalidade por km)'); ax.set_ylabel('quebras por km', color='#1F4E79')
ax.set_title('(a) Funcional: IGG l2 - calibracao de alpha_km')

ax = axes[0, 1]
ax.step(GRID_BETA, K_REDE, where='post', color='#1F4E79', lw=1.3)
ax.axvline(BETA_COTOVELO_REDE, color='#C65A57', lw=1.3, label=f'cotovelo rede: beta={BETA_COTOVELO_REDE:.2f}')
ax.axvline(BETA_BIC, color='#2E7D32', ls=':', lw=1.3, label=f'BIC 3 ln(N): {BETA_BIC:.2f}')
if BETA_FIXO is not None:
    ax.axvline(BETA_REDE, color='#555555', ls='--', lw=1.3, label=f'BETA_FIXO: {BETA_REDE:.2f}')
ax.set_xlabel('penalidade beta'); ax.set_ylabel('n. de quebras na malha')
ax.set_title('(b) Estrutural: diagnostico da rede (d0, PELT normal, CROPS)')
ax.legend(fontsize=8)

ax = axes[1, 0]
ref = SRE_REFERENCIA if SRE_REFERENCIA in CAMINHOS else (max(CAMINHOS, key=lambda s: len(grupos[s]['ini'])) if CAMINHOS else None)
if ref is not None:
    c = CAMINHOS[ref]
    g = np.linspace(BETA_MIN_EF, max(c['sup'][1] * 1.1, BETA_MIN_EF + 1), N_GRID_BETA) if len(c['Ks']) > 1 else GRID_BETA
    ax.step(g, k_otimo(c['Ks'], c['Q'], g), where='post', color='#1F4E79', lw=1.3)
    ax.plot(c['inf'][1:], c['Ks'][1:], 'o', color='#1F4E79', ms=3)
    if COTOVELO.get(ref) is not None:
        ax.axvline(COTOVELO[ref][1], color='#555555', ls='--', lw=1.2,
                   label=f'cotovelo SRE: beta={COTOVELO[ref][1]:.2f} (K={COTOVELO[ref][0]})')
    ax.axvline(BETA_REDE, color='#C65A57', lw=1.2, label=f'beta rede: {BETA_REDE:.2f}')
    ax.set_title(f'(c) Diagnostico do SRE {ref} (B&K Fig. 9)'); ax.legend(fontsize=8)
ax.set_xlabel('penalidade beta'); ax.set_ylabel('n. de quebras')

ax = axes[1, 1]
if len(b_cot):
    ax.hist(np.clip(b_cot, None, BETA_MAX_REDE * 1.5), bins=40, color='#9DB4CC', edgecolor='white')
    ax.axvline(BETA_REDE, color='#C65A57', lw=1.3, label=f'beta rede: {BETA_REDE:.2f}')
    ax.axvline(np.median(b_cot), color='#555555', ls='--', lw=1.2, label=f'mediana SREs: {np.median(b_cot):.2f}')
    ax.legend(fontsize=8)
ax.set_xlabel('beta do cotovelo de cada SRE'); ax.set_ylabel('n. de SREs')
ax.set_title('(d) Dispersao do cotovelo individual')
for ax in axes.ravel():
    ax.grid(True, ls=':', alpha=0.5)
plt.tight_layout()
CALIB_PNG = os.path.join(OUTPUT_DIR, '_calibracao.png')
plt.savefig(CALIB_PNG, dpi=DPI, bbox_inches='tight'); plt.show()
print(f"Calibracao: funcional {TEMPOS['calib_funcional']:.1f}s | estrutural {TEMPOS['calib_estrutural']:.1f}s")

In [ ]:
# ─── 6. Funcoes de Segmentacao por SRE (iguais a v8) ──────────────────────────
def atributos(a, lo, hi, km_a, km_b):
    d0 = a['d0'][lo:hi]
    igg = a['igg'][lo:hi]
    iri = a['iri'][lo:hi]
    d0m, d0cv   = media_cv(d0)
    iggm, iggcv = media_cv(igg)
    irim, iricv = media_cv(iri)
    cvs = [c for c in (d0cv, iggcv, iricv) if c is not None and not (isinstance(c, float) and np.isnan(c))]
    coerente = bool(cvs) and max(cvs) <= CV_LIMITE
    d0sd  = float(np.std(d0, ddof=1)) if len(d0) > 1 and np.isfinite(d0).all() else None
    iggsd = float(np.std(igg, ddof=1)) if len(igg) > 1 and np.isfinite(igg).all() else None
    irisd = float(np.std(iri, ddof=1)) if len(iri) > 1 and np.isfinite(iri).all() else None
    return {
        'km_inicial': round(km_a, 3), 'km_final': round(km_b, 3),
        'comprimento_km': round(km_b - km_a, 3), 'n_intervalos': int(hi - lo),
        'd0_medio': round(d0m, 2) if d0m is not None and np.isfinite(d0m) else None,
        'd0_desvio': round(d0sd, 2) if d0sd is not None else None,
        'd0_erro_padrao': round(d0sd / np.sqrt(hi - lo), 2) if d0sd is not None else None,   # B&K Tab. 2
        'd0_cv': round(d0cv, 3) if d0cv is not None and np.isfinite(d0cv) else None,
        'd0_frac_medido': round(float(a['d0_obs'][lo:hi].mean()), 3),
        'igg_medio': round(iggm, 2) if iggm is not None and np.isfinite(iggm) else None,
        'igg_desvio': round(iggsd, 2) if iggsd is not None else None,
        'igg_cv': round(iggcv, 3) if iggcv is not None and np.isfinite(iggcv) else None,
        'iri_medio': round(irim, 2) if irim is not None and np.isfinite(irim) else None,
        'iri_desvio': round(irisd, 2) if irisd is not None else None,
        'iri_cv': round(iricv, 3) if iricv is not None and np.isfinite(iricv) else None,
        'coerente': coerente,
    }

def montar_segmentos(tipo, sre, inicios, a, km_min, km_max):
    N = len(a['ini'])
    idx = [0] + sorted(inicios) + [N]
    kms = [km_min] + [float(a['ini'][s]) for s in sorted(inicios)] + [km_max]
    linhas = []
    for i in range(len(idx) - 1):
        row = {'tipo': tipo, 'sre': sre, 'seg_id': i + 1}
        row.update(atributos(a, idx[i], idx[i + 1], kms[i], kms[i + 1]))
        linhas.append(row)
    return linhas

def segmentar_sre(sre, a, alpha_func, beta, origem_beta, cam, cotovelo):
    N = len(a['ini'])
    ini = a['ini']
    km_min, km_max = float(ini.min()), float(a['fim'].max())
    tem_igg = bool(np.isfinite(a['igg']).all())
    tem_d0 = cam is not None
    vf = piso_variancia(a['d0'])

    # Funcional (v6): PELT l2 sobre IGG
    func_s, pen_func, _ = segmentar_funcional_sre(a, alpha_func) if tem_igg else ([], None, None)
    n_func_bruto = len(func_s)
    if FUNDIR_FUNC and tem_igg:
        func_s = fundir_testes(func_s, [(a['igg'], False)], N)

    # Estrutural (B&K): solucao do caminho CROPS no beta escolhido
    if tem_d0:
        K = int(k_otimo(cam['Ks'], cam['Q'], [beta])[0])
        estrut_s = [int(b) for b in cam['bkps'][K][:-1]]
    else:
        estrut_s = []
    n_estrut_bruto = len(estrut_s)
    if FUNDIR_ESTRUT and tem_d0:
        estrut_s = fundir_testes(estrut_s, [(a['d0'], True)], N)

    # Combinada (2 etapas): bases funcionais -> PELT normal (mesmo beta) dentro de cada base
    if tem_d0 and SUBSEGMENTAR_COMB:
        lims = [0] + func_s + [N]
        sub_s = []
        for i in range(len(lims) - 1):
            sub_s.extend(segmentar_estrutural_trecho(a['d0'], lims[i], lims[i + 1], beta, vf))
    else:
        sub_s = estrut_s
    func_km = [float(ini[s]) for s in func_s]
    comb_km = reconciliar_combinada(func_km, [float(ini[s]) for s in sub_s], km_min, km_max)
    comb_s = [int(i) for i in np.searchsorted(ini, comb_km)]
    n_comb_bruto = len(comb_s)
    if FUNDIR_COMB:
        series = ([(a['d0'], True)] if tem_d0 else []) + ([(a['igg'], False)] if tem_igg else [])
        if series:
            comb_s = fundir_testes(comb_s, series, N)

    # Auditoria de residuos da segmentacao estrutural (B&K, "Test of Assumptions")
    if tem_d0:
        aud, z = auditar_residuos(a['d0'], a['d0_obs'], estrut_s)
    else:
        aud, z = {'n_residuos': 0, 'n_segmentos': 1}, None
    aud = {'sre': sre, 'beta': round(float(beta), 4) if tem_d0 else None, **aud}

    linhas = (montar_segmentos('funcional', sre, func_s, a, km_min, km_max)
              + montar_segmentos('estrutural', sre, estrut_s, a, km_min, km_max)
              + montar_segmentos('combinada', sre, comb_s, a, km_min, km_max))
    info = {'sre': sre, 'n_intervalos': N, 'extensao_km': round(km_max - km_min, 3),
            'frac_d0_medido': round(float(a['d0_obs'].mean()), 3),
            'pen_func': pen_func,
            'beta_estrut': round(float(beta), 4) if tem_d0 else None, 'origem_beta': origem_beta if tem_d0 else 'sem_d0',
            'beta_cotovelo_sre': round(cotovelo[1], 4) if cotovelo else None,
            'K_cotovelo_sre': cotovelo[0] if cotovelo else None,
            'beta_zero_quebras': round(float(cam['sup'][1]), 4) if tem_d0 and len(cam['Ks']) > 1 else None,
            'n_solucoes_crops': len(cam['Ks']) if tem_d0 else 0,
            'piso_var_d0': round(vf, 4) if tem_d0 else None,
            'quebras_func_pelt': n_func_bruto, 'quebras_func': len(func_s),
            'quebras_estrut_pelt': n_estrut_bruto, 'quebras_estrut': len(estrut_s),
            'quebras_comb_pelt': n_comb_bruto, 'quebras_comb': len(comb_s)}
    plot = {'func_s': func_s, 'estrut_s': estrut_s, 'comb_s': comb_s,
            'func_km': func_km, 'estrut_km': [float(ini[s]) for s in estrut_s],
            'comb_km': [float(ini[s]) for s in comb_s], 'beta': float(beta) if tem_d0 else None,
            'cotovelo': cotovelo, 'z': z, 'aud': aud,
            'cam': {k: cam[k] for k in ('Ks', 'Q', 'inf', 'sup')} if tem_d0 else None}
    return linhas, info, plot

print("OK funcoes de segmentacao")


In [ ]:
# ─── 7. Analise de Sensibilidade: SREs analisados com a penalidade escalada ───
# Cada cenario repete a segmentacao da v8 com a penalidade do alvo multiplicada por um fator;
# a outra penalidade fica na calibrada. Fator 1.00 = resultado da v8 (referencia).
FATORES  = sorted(set(float(f) for f in SENS_FATORES) | {1.0})
I_REF    = FATORES.index(1.0)
NOME_PEN = 'beta' if SENS_ALVO == 'estrutural' else 'alpha_km'
PEN_REF  = BETA_REDE if SENS_ALVO == 'estrutural' else ALPHA_KM_FUNC_OPT
CORES_F  = plt.cm.viridis(np.linspace(0.05, 0.90, len(FATORES)))
TIPOS    = ['funcional', 'estrutural', 'combinada']
AB       = {'funcional': 'func', 'estrutural': 'estrut', 'combinada': 'comb'}
CHAVE_KM = {t: f'{AB[t]}_km' for t in TIPOS}
SRES_SENS = sres if SRES_ANALISE is None else [s for s in SRES_ANALISE if s in grupos]
assert SRES_SENS, f'nenhum SRE de SRES_ANALISE encontrado: {SRES_ANALISE}'
SRES_TIPO = {'funcional': SRES_SENS, 'estrutural': [s for s in SRES_SENS if s in CAMINHOS], 'combinada': SRES_SENS}
EXT_TOT   = {t: sum(EXT[s] for s in SRES_TIPO[t]) for t in TIPOS}

def cenarios_sre(sre, a, fatores, alpha_func, beta, origem_beta, cam, cotovelo):
    out = []
    for f in fatores:
        al, be = (alpha_func, beta * f) if SENS_ALVO == 'estrutural' else (alpha_func * f, beta)
        out.append(segmentar_sre(sre, a, al, be, origem_beta, cam, cotovelo))
    return out

def concordancia(ref_km, km, tol):
    """(quebras da referencia com uma quebra do cenario a <= tol, quebras do cenario com uma da referencia a <= tol)."""
    r, c = np.asarray(ref_km, float), np.asarray(km, float)
    if len(r) == 0 or len(c) == 0:
        return 0, 0
    d = np.abs(r[:, None] - c[None, :])
    return int((d.min(axis=1) <= tol).sum()), int((d.min(axis=0) <= tol).sum())

def _media_pond(seg, col):
    s = seg[[col, 'comprimento_km']].dropna()
    return round(float(np.average(s[col], weights=s['comprimento_km'])), 4) if s['comprimento_km'].sum() > 0 else np.nan

t0 = time.time()
_res = Parallel(n_jobs=N_JOBS, batch_size=8)(
    delayed(cenarios_sre)(s, grupos[s], FATORES, ALPHA_KM_FUNC_OPT, BETA_SRE.get(s, BETA_REDE),
                          ORIGEM_BETA.get(s, 'rede'), CAMINHOS.get(s), COTOVELO.get(s)) for s in SRES_SENS)
seg_l, info_l, aud_l, PLOTS = [], [], [], {}
for s, rs in zip(SRES_SENS, _res):
    PLOTS[s] = [r[2] for r in rs]
    for f, (linhas, info, plot) in zip(FATORES, rs):
        seg_l += [{'fator': f, **l} for l in linhas]
        info_l.append({'fator': f, **info})
        aud_l.append({'fator': f, **plot['aud']})
del _res
df_seg_s  = pd.DataFrame(seg_l)
df_info_s = pd.DataFrame(info_l)
df_aud_s  = pd.DataFrame(aud_l)
TEMPOS['sensibilidade'] = time.time() - t0

# ── Metricas da rede por cenario ──
linhas_sens = []
for i, f in enumerate(FATORES):
    info = df_info_s[df_info_s['fator'] == f].set_index('sre')
    row = {'fator': f, NOME_PEN: round(PEN_REF * f, 4), 'referencia': i == I_REF}
    for t in TIPOS:
        ab, ss = AB[t], SRES_TIPO[t]
        seg = df_seg_s[(df_seg_s['fator'] == f) & (df_seg_s['tipo'] == t) & df_seg_s['sre'].isin(ss)]
        nr = nc = rec = pre = ident = 0
        for s in ss:
            kr, kc = PLOTS[s][I_REF][CHAVE_KM[t]], PLOTS[s][i][CHAVE_KM[t]]
            r_, p_ = concordancia(kr, kc, SENS_TOL_KM)
            nr += len(kr); nc += len(kc); rec += r_; pre += p_; ident += int(list(kr) == list(kc))
        R = rec / nr if nr else 1.0
        P = pre / nc if nc else 1.0
        comp = seg['comprimento_km']
        row.update({
            f'{ab}_segmentos': int(len(seg)),
            f'{ab}_quebras_por_km': round(int(info.loc[ss, f'quebras_{ab}'].sum()) / max(EXT_TOT[t], 1e-9), 4),
            f'{ab}_comp_mediano_km': round(float(comp.median()), 3),
            f'{ab}_comp_p10_km': round(float(comp.quantile(0.10)), 3),
            f'{ab}_comp_p90_km': round(float(comp.quantile(0.90)), 3),
            f'{ab}_pct_coerentes': round(100 * float(seg['coerente'].mean()), 1),
            f'{ab}_d0_cv_pond': _media_pond(seg, 'd0_cv'),
            f'{ab}_igg_cv_pond': _media_pond(seg, 'igg_cv'),
            f'{ab}_recall_ref': round(R, 4), f'{ab}_precisao_ref': round(P, 4),
            f'{ab}_f1_ref': round(2 * R * P / (R + P), 4) if R + P > 0 else 0.0,
            f'{ab}_pct_sres_identicos': round(100 * ident / max(len(ss), 1), 1),
        })
    aud = df_aud_s[df_aud_s['fator'] == f].dropna(subset=['shapiro_p'])
    row.update({
        'aud_sres': int(len(aud)),
        'aud_pct_normalidade_ok': round(100 * float(aud['normalidade_ok'].mean()), 1) if len(aud) else np.nan,
        'aud_pct_independencia_ok': round(100 * float(aud['independencia_ok'].mean()), 1) if len(aud) else np.nan,
        'aud_acf_lag1_mediana': round(float(aud['acf_lag1'].median()), 4) if len(aud) else np.nan,
        'aud_pct_heterocedastico': round(100 * float(aud['heterocedastico'].dropna().mean()), 1) if len(aud) else np.nan,
    })
    linhas_sens.append(row)
SENS_REDE = pd.DataFrame(linhas_sens)

# ── Metricas por SRE: cada SRE responde de forma propria a penalidade ──
# faixa estavel (estrutural) = intervalo de beta do caminho CROPS em que a solucao da referencia continua otima;
# quando a referencia e a solucao mais fina do caminho, o limite inferior e so o inicio da varredura (BETA_MIN_EF).
def _f1(kr, kc, tol):
    r_, p_ = concordancia(kr, kc, tol)
    R = r_ / len(kr) if len(kr) else 1.0
    P = p_ / len(kc) if len(kc) else 1.0
    return round(2 * R * P / (R + P), 4) if R + P > 0 else 0.0

linhas_sre = []
for s in SRES_SENS:
    ps = PLOTS[s]
    row = {'sre': s, 'n_intervalos': len(grupos[s]['ini']), 'extensao_km': round(EXT[s], 3), 'tem_d0': s in CAMINHOS}
    if SENS_ALVO == 'estrutural' and s in CAMINHOS:
        c = CAMINHOS[s]; b = BETA_SRE.get(s, BETA_REDE)
        j = int(np.flatnonzero(c['Ks'] == k_otimo(c['Ks'], c['Q'], [b])[0])[0])
        sup_ = float(c['sup'][j])
        row.update({'beta_ref': round(b, 4), 'K_ref': int(c['Ks'][j]),
                    'beta_estavel_inf': round(float(c['inf'][j]), 4),
                    'beta_estavel_sup': round(sup_, 4) if np.isfinite(sup_) else None,
                    'fator_estavel_inf': round(float(c['inf'][j]) / b, 3),
                    'fator_estavel_sup': round(sup_ / b, 3) if np.isfinite(sup_) else None})
    for t in TIPOS:
        k = CHAVE_KM[t]
        for i, f in enumerate(FATORES):
            row[f'{AB[t]}_seg_{f:g}x'] = len(ps[i][k]) + 1
        for i, f in enumerate(FATORES):
            if i != I_REF:
                row[f'{AB[t]}_f1_{f:g}x'] = _f1(ps[I_REF][k], ps[i][k], SENS_TOL_KM)
    linhas_sre.append(row)
SENS_POR_SRE = pd.DataFrame(linhas_sre)
ESTAVEL = {r['sre']: (r['beta_estavel_inf'], r['beta_estavel_sup'], r['K_ref'])
           for r in linhas_sre if 'beta_estavel_inf' in r}

print(f"OK {len(FATORES)} cenarios x {len(SRES_SENS)} SRE(s) em {TEMPOS['sensibilidade']:.1f}s | alvo={SENS_ALVO} | "
      f"{NOME_PEN} referencia={PEN_REF:.4f}" + (" (modo 'sre': fator aplicado ao beta de cada SRE)"
                                               if SENS_ALVO == 'estrutural' and MODO_PENALIDADE == 'sre' else ''))
_cols = ['fator', NOME_PEN] + [f'{AB[t]}_{m}' for t in TIPOS for m in ('segmentos', 'comp_mediano_km', 'f1_ref')]
print(SENS_REDE[_cols].to_string(index=False))
if 'fator_estavel_inf' in SENS_POR_SRE:
    _e = SENS_POR_SRE.dropna(subset=['fator_estavel_inf'])
    print(f"Faixa estavel por SRE ({len(_e)} SREs com d0): fator inferior mediano={_e['fator_estavel_inf'].median():.2f}x | "
          f"superior mediano={_e['fator_estavel_sup'].median():.2f}x")


In [ ]:
# ─── 8. Graficos de Sensibilidade (agregado dos SREs analisados) ──────────────
x = np.array(FATORES)
def _eixo_fator(ax):
    ax.set_xscale('log'); ax.set_xticks(x); ax.set_xticklabels([f'{f:g}x' for f in x], fontsize=8)
    ax.minorticks_off(); ax.axvline(1.0, color='#C65A57', lw=1.0, alpha=0.6)
    ax.set_xlabel(f'fator sobre {NOME_PEN} calibrado ({PEN_REF:.4g})')

# ── Figura 1: resposta da segmentacao a penalidade ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax = axes[0, 0]
if SENS_ALVO == 'estrutural':
    ax.step(GRID_BETA, K_REDE, where='post', color='#1F4E79', lw=1.3)
    ax.axvline(BETA_BIC, color='#2E7D32', ls=':', lw=1.3, label=f'BIC 3 ln(N): {BETA_BIC:.2f}')
    ax.set_xlabel('penalidade beta'); ax.set_ylabel('n. de quebras na malha (estrutural, sem fusao)')
    ax.set_title('(a) Penalidades testadas sobre o diagnostico da rede (CROPS)')
elif CURVA_FUNC is not None:
    ax.plot(CURVA_FUNC['alpha_km'], CURVA_FUNC['quebras_por_km'], 'o-', color='#1F4E79', ms=3)
    ax.set_xlabel('alpha_km (penalidade por km)'); ax.set_ylabel('quebras por km (media, calibracao)')
    ax.set_title('(a) Penalidades testadas sobre a curva de calibracao funcional')
for i, f in enumerate(FATORES):
    ax.axvline(PEN_REF * f, color=CORES_F[i], lw=2.2 if i == I_REF else 1.2,
               label=f'{f:g}x = {PEN_REF * f:.3g}' + (' (ref)' if i == I_REF else ''))
ax.legend(fontsize=7.5, ncol=2)

ax = axes[0, 1]
for t, cor, ls in [('funcional', '#555555', '--'), ('estrutural', '#1F4E79', ':'), ('combinada', '#C65A57', '-')]:
    ax.plot(x, SENS_REDE[f'{AB[t]}_quebras_por_km'], 'o', ls=ls, color=cor, lw=1.4, ms=4, label=t)
_eixo_fator(ax); ax.set_ylabel('quebras por km'); ax.legend(fontsize=8)
ax.set_title('(b) Densidade de quebras na malha')

ax = axes[1, 0]
_dados = [df_seg_s[(df_seg_s['fator'] == f) & (df_seg_s['tipo'] == SENS_ALVO)
                   & df_seg_s['sre'].isin(SRES_TIPO[SENS_ALVO])]['comprimento_km'].to_numpy() for f in FATORES]
bp = ax.boxplot(_dados, showfliers=False, patch_artist=True, widths=0.6)
for patch, cor in zip(bp['boxes'], CORES_F):
    patch.set_facecolor(cor); patch.set_alpha(0.55)
for med in bp['medians']:
    med.set_color('black')
ax.set_xticks(range(1, len(FATORES) + 1)); ax.set_xticklabels([f'{f:g}x' for f in FATORES], fontsize=8)
ax.axhline(MIN_COMP_COMB_KM, color='#555555', ls='--', lw=1.0, label=f'MIN_COMP_COMB_KM={MIN_COMP_COMB_KM}')
ax.set_xlabel(f'fator sobre {NOME_PEN}'); ax.set_ylabel('comprimento do segmento (km)'); ax.legend(fontsize=8)
ax.set_title(f'(c) Comprimento dos segmentos ({SENS_ALVO}, sem outliers)')

ax = axes[1, 1]
for t, cor, ls in [(SENS_ALVO, '#1F4E79', ':'), ('combinada', '#C65A57', '-')]:
    ax.plot(x, SENS_REDE[f'{AB[t]}_d0_cv_pond'], 'o', ls=ls, color=cor, lw=1.3, ms=4, label=f'CV d0 ({t})')
    ax.plot(x, SENS_REDE[f'{AB[t]}_igg_cv_pond'], 's', ls=ls, color=cor, lw=1.0, ms=3.5, alpha=0.6, label=f'CV IGG ({t})')
_eixo_fator(ax); ax.set_ylabel('CV medio ponderado por comprimento')
ax2 = ax.twinx()
ax2.plot(x, SENS_REDE[f'{AB[SENS_ALVO]}_pct_coerentes'], 'D--', color='#B8860B', lw=1.0, ms=3.5, label=f'% coerentes ({SENS_ALVO})')
ax2.set_ylabel('% segmentos coerentes (CV <= CV_LIMITE)', color='#B8860B')
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=7.5)
ax.set_title('(d) Homogeneidade interna dos segmentos')
for ax in axes.ravel():
    ax.grid(True, ls=':', alpha=0.5)
plt.tight_layout()
SENS_PNG = os.path.join(OUTPUT_DIR, '_sensibilidade_rede.png')
plt.savefig(SENS_PNG, dpi=DPI, bbox_inches='tight'); plt.show()

# ── Figura 2: estabilidade das quebras e premissas do modelo ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax = axes[0, 0]
for t, cor, ls in [('funcional', '#555555', '--'), ('estrutural', '#1F4E79', ':'), ('combinada', '#C65A57', '-')]:
    ax.plot(x, SENS_REDE[f'{AB[t]}_f1_ref'], 'o', ls=ls, color=cor, lw=1.4, ms=4, label=f'F1 {t}')
    ax.plot(x, SENS_REDE[f'{AB[t]}_pct_sres_identicos'] / 100, 'x', ls=ls, color=cor, lw=0.8, alpha=0.55,
            label=f'SREs identicos {t}')
_eixo_fator(ax); ax.set_ylim(-0.03, 1.03); ax.set_ylabel('fracao')
ax.legend(fontsize=7, ncol=2)
ax.set_title(f'(a) Concordancia das quebras com a referencia (tol. {SENS_TOL_KM} km)')

ax = axes[0, 1]
for col, cor, rot in [('aud_pct_normalidade_ok', '#1F4E79', 'normalidade (SW e KS)'),
                      ('aud_pct_independencia_ok', '#2E7D32', 'independencia (Ljung-Box)'),
                      ('aud_pct_heterocedastico', '#C65A57', 'heterocedastico (B-F)')]:
    ax.plot(x, SENS_REDE[col], 'o-', color=cor, lw=1.3, ms=4, label=rot)
_eixo_fator(ax); ax.set_ylabel(f'% SREs (p > {ALPHA_TESTE}; B-F: p <= {ALPHA_TESTE})')
ax2 = ax.twinx()
ax2.plot(x, SENS_REDE['aud_acf_lag1_mediana'], 's--', color='#B8860B', lw=1.0, ms=3.5, label='ACF lag 1 (mediana)')
ax2.set_ylabel('ACF lag 1 mediana', color='#B8860B')
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=7.5)
ax.set_title('(b) Premissas dos residuos estruturais (B&K Tab. 1)')

ax = axes[1, 0]
_q = []
for f in FATORES:
    inf_ = df_info_s[(df_info_s['fator'] == f) & df_info_s['sre'].isin(SRES_TIPO[SENS_ALVO])].set_index('sre')
    _q.append((inf_[f'quebras_{AB[SENS_ALVO]}'] / pd.Series(EXT).reindex(inf_.index).clip(lower=1e-9)).to_numpy())
bp = ax.boxplot(_q, showfliers=False, patch_artist=True, widths=0.6)
for patch, cor in zip(bp['boxes'], CORES_F):
    patch.set_facecolor(cor); patch.set_alpha(0.55)
for med in bp['medians']:
    med.set_color('black')
ax.set_xticks(range(1, len(FATORES) + 1)); ax.set_xticklabels([f'{f:g}x' for f in FATORES], fontsize=8)
ax.set_xlabel(f'fator sobre {NOME_PEN}'); ax.set_ylabel('quebras por km no SRE')
ax.set_title(f'(c) Dispersao entre SREs ({SENS_ALVO}, sem outliers)')

ax = axes[1, 1]
_rng = np.random.default_rng(0)
for i, f in enumerate(FATORES):
    zr = np.concatenate([PLOTS[s][i]['z'][np.isfinite(PLOTS[s][i]['z'])] for s in SRES_SENS if PLOTS[s][i]['z'] is not None])
    if len(zr) >= 3:
        am = np.sort(_rng.choice(zr, size=min(4000, len(zr)), replace=False))
        osm = stats.norm.ppf((np.arange(1, len(am) + 1) - 0.5) / len(am))
        ax.plot(osm, am, color=CORES_F[i], lw=2.0 if i == I_REF else 1.1, label=f'{f:g}x')
lim_ = np.array([-4, 4]); ax.plot(lim_, lim_, color='#C65A57', lw=1.0, ls='--')
ax.set_xlabel('quantis teoricos N(0,1)'); ax.set_ylabel('residuos padronizados z')
ax.legend(fontsize=7.5, ncol=2); ax.set_title('(d) Q-Q dos residuos padronizados na malha (B&K Fig. 11)')
for ax in axes.ravel():
    ax.grid(True, ls=':', alpha=0.5)
plt.tight_layout()
ESTAB_PNG = os.path.join(OUTPUT_DIR, '_sensibilidade_estabilidade.png')
plt.savefig(ESTAB_PNG, dpi=DPI, bbox_inches='tight'); plt.show()


In [ ]:
# ─── 9. Sequencia Explicativa por SRE: penalidade crescente (paralelo, Agg) ───
# Cada SRE ganha uma pasta com um grafico por penalidade, da mais fraca (muitas quebras) a mais forte (nenhuma).
# Grade 'crops': cada passo e uma solucao distinta do caminho CROPS do SRE, entao de um grafico para o seguinte
# a segmentacao estrutural sempre muda, e o quadro de texto explica o que mudou e por que.
COR_REMOVIDA, COR_NOVA = '#9E9E9E', '#2E7D32'

def passos_sre(sre, a, alpha_ref, beta_ref, origem, cam, cotovelo):
    """Penalidades crescentes do SRE -> lista de plots de segmentar_sre (com 'alpha', 'beta', 'fator')."""
    if SENS_ALVO == 'estrutural' and SENS_GRADE == 'crops' and cam is not None and len(cam['Ks']) > 1:
        Ks, inf_, sup = cam['Ks'], cam['inf'], cam['sup']
        K_ref = int(k_otimo(Ks, cam['Q'], [beta_ref])[0])
        betas = []
        for j in range(len(Ks) - 1, -1, -1):          # K decrescente = beta crescente
            if Ks[j] == K_ref:
                betas.append(float(beta_ref))          # o passo da referencia usa o beta da v8
            elif np.isfinite(sup[j]):
                betas.append(0.5 * (inf_[j] + sup[j]))  # centro do intervalo em que a solucao e otima
            else:
                betas.append(1.25 * inf_[j])            # K = 0: qualquer beta acima de inf
        if len(betas) > SENS_MAX_PASSOS:
            i_r = betas.index(float(beta_ref))
            idx = np.unique(np.r_[np.linspace(0, len(betas) - 1, SENS_MAX_PASSOS).round().astype(int), i_r])
            betas = [betas[i] for i in idx]
        cen = [(alpha_ref, b) for b in betas]
    elif SENS_ALVO == 'estrutural':
        cen = [(alpha_ref, beta_ref * f) for f in FATORES]
    else:
        cen = [(alpha_ref * f, beta_ref) for f in FATORES]
    out = []
    for al, be in cen:
        p = segmentar_sre(sre, a, al, be, origem, cam, cotovelo)[2]
        p.update({'alpha': al, 'beta': be if cam is not None else None,
                  'fator': (be / beta_ref) if SENS_ALVO == 'estrutural' else (al / alpha_ref)})
        out.append(p)
    return out

def _km_lim(a, lo, hi):
    return float(a['ini'][lo]), (float(a['ini'][hi]) if hi < len(a['ini']) else float(a['fim'][-1]))

def _desc(a, serie, lo, hi, un):
    v = serie[lo:hi]
    k0, k1 = _km_lim(a, lo, hi)
    sd = float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
    return f"km {k0:.1f}-{k1:.1f} (media {np.mean(v):.1f}, desvio {sd:.1f}{un})"

def explicar_passo(a, p, prev, cam, alvo, nome_pen):
    """Texto do passo: o que o PELT otimiza, o que mudou desde o passo anterior e por que compensou."""
    chave = 'estrut_s' if alvo == 'estrutural' else 'func_s'
    serie, un = (a['d0'], ' um') if alvo == 'estrutural' else (a['igg'], '')
    pen = p['beta'] if alvo == 'estrutural' else p['alpha']
    L = []
    if alvo == 'estrutural':
        L.append(f"PELT normal sobre d0 minimiza  Q + beta x K,  com Q = soma de n.log(var) dos segmentos "
                 f"(-2 log-verossimilhanca) e K = n. de quebras.")
        if cam is not None and len(cam['Ks']) > 1:
            K = len(p['estrut_s'])
            j = int(np.flatnonzero(cam['Ks'] == K)[0]) if K in cam['Ks'] else None
            if j is not None:
                sup_ = cam['sup'][j]
                L.append(f"Com beta = {pen:.2f}, a melhor solucao tem K = {K} quebras (Q = {cam['Q'][j]:.1f}); ela continua "
                         f"otima para beta entre {cam['inf'][j]:.2f} e " + (f"{sup_:.2f}." if np.isfinite(sup_) else "infinito."))
    else:
        L.append(f"PELT l2 sobre IGG padronizado; penalidade = alpha_km x extensao, escolhida pelo BIC numa faixa de "
                 f"+-{int(MARGEM_PEN * 100)}% (alpha_km = {pen:.4f}).")
    if prev is None:
        L.append("Primeiro passo: penalidade mais fraca da sequencia, maior numero de quebras.")
    else:
        rem = sorted(set(prev[chave]) - set(p[chave])); nov = sorted(set(p[chave]) - set(prev[chave]))
        pen0 = prev['beta'] if alvo == 'estrutural' else prev['alpha']
        L.append(f"Desde o passo anterior ({nome_pen} {pen0:.3g} -> {pen:.3g}): {len(prev[chave])} -> {len(p[chave])} quebras "
                 f"({len(rem)} removidas, {len(nov)} novas).")
        if alvo == 'estrutural' and cam is not None and len(cam['Ks']) > 1:
            K0, K1 = len(prev['estrut_s']), len(p['estrut_s'])
            if K0 in cam['Ks'] and K1 in cam['Ks'] and K0 != K1:
                Q0 = float(cam['Q'][cam['Ks'] == K0][0]); Q1 = float(cam['Q'][cam['Ks'] == K1][0])
                L.append(f"Por que: com {K1} quebras o ajuste piora (Q sobe {Q1 - Q0:.1f}), mas a penalidade economiza "
                         f"beta x {K0 - K1} = {pen * (K0 - K1):.1f}; a troca compensa a partir de beta = {(Q1 - Q0) / (K0 - K1):.2f}.")
        lim0 = [0] + sorted(prev[chave]) + [len(serie)]
        for b in rem[:4]:
            i = lim0.index(b)
            L.append(f"  - removida em km {float(a['ini'][b]):.1f}: unia {_desc(a, serie, lim0[i - 1], b, un)} "
                     f"e {_desc(a, serie, b, lim0[i + 1], un)}")
        lim1 = [0] + sorted(p[chave]) + [len(serie)]
        for b in nov[:3]:
            i = lim1.index(b)
            L.append(f"  + nova em km {float(a['ini'][b]):.1f}: separa {_desc(a, serie, lim1[i - 1], b, un)} "
                     f"de {_desc(a, serie, b, lim1[i + 1], un)}")
        if len(rem) > 4 or len(nov) > 3:
            L.append(f"  (... {max(len(rem) - 4, 0)} removidas e {max(len(nov) - 3, 0)} novas nao listadas)")
        L.append(f"Combinada: {len(prev['comb_s']) + 1} -> {len(p['comb_s']) + 1} segmentos.")
    sw, lb = p['aud'].get('shapiro_p'), p['aud'].get('ljung_box_p')
    if sw is not None and np.isfinite(sw):
        L.append(f"Residuos estruturais: Shapiro p = {sw:.3f} ({'normal ok' if sw > ALPHA_TESTE else 'normalidade rejeitada'}) | "
                 f"Ljung-Box p = {lb:.3f} ({'independentes' if lb > ALPHA_TESTE else 'autocorrelacionados: quebras de menos'}).")
    return L

def plot_passo(sre, a, p, prev, i, n, i_ref, output_dir, dpi, beta_min, alvo, nome_pen):
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg

    km_c = a['centro']; d0 = a['d0']; igg = a['igg']; iri = a['iri']
    chave = 'estrut_s' if alvo == 'estrutural' else 'func_s'
    rem_km = [float(a['ini'][b]) for b in sorted(set(prev[chave]) - set(p[chave]))] if prev else []
    nov_km = [float(a['ini'][b]) for b in sorted(set(p[chave]) - set(prev[chave]))] if prev else []

    fig = Figure(figsize=(14, 12))
    FigureCanvasAgg(fig)
    gs = fig.add_gridspec(4, 2, height_ratios=[1.2, 0.8, 1.0, 0.95], hspace=0.38, wspace=0.22)
    ax_d0  = fig.add_subplot(gs[0, :])
    ax_igg = fig.add_subplot(gs[1, :], sharex=ax_d0)
    ax_cr  = fig.add_subplot(gs[2, 0])
    ax_tr  = fig.add_subplot(gs[2, 1])
    ax_tx  = fig.add_subplot(gs[3, :]); ax_tx.axis('off')

    # (1) d0 + media e +-1 desvio por secao estrutural deste passo
    ax_d0.plot(km_c, d0, color='#333333', lw=0.8, alpha=0.75)
    lim = [0] + p['estrut_s'] + [len(d0)]
    for j in range(len(lim) - 1):
        lo, hi = lim[j], lim[j + 1]
        seg = d0[lo:hi]
        if len(seg) and np.isfinite(seg).all():
            m = float(seg.mean()); sd = float(seg.std(ddof=1)) if len(seg) > 1 else 0.0
            x0, x1 = _km_lim(a, lo, hi)
            ax_d0.hlines(m, x0, x1, color='#C65A57', lw=2.0)
            ax_d0.fill_between([x0, x1], m - sd, m + sd, color='#C65A57', alpha=0.15)
    ax_d0.set_ylabel('d0 (um)')

    # (2) IGG + IRI
    ax_igg.plot(km_c, igg, color='#C65A57', lw=0.95)
    ax_igg.set_ylabel('IGG', color='#C65A57'); ax_igg.set_xlabel('Distancia Linear (km)')
    ax_iri = ax_igg.twinx()
    ax_iri.plot(km_c, iri, color='#2E7D32', lw=0.8, alpha=0.65)
    ax_iri.set_ylabel('IRI (m/km)', color='#2E7D32')

    for ax in (ax_d0, ax_igg):
        ax.grid(True, linestyle=':', alpha=0.5)
        for f in p['func_km']:
            ax.axvline(f, color='#555555', ls='--', lw=0.9, alpha=0.8)
        for f in p['estrut_km']:
            ax.axvline(f, color='#1F4E79', ls=':', lw=1.2, alpha=0.85)
        for f in p['comb_km']:
            ax.axvline(f, color='#C65A57', ls='-', lw=1.1, alpha=0.75)
        for f in rem_km:
            ax.axvline(f, color=COR_REMOVIDA, ls=(0, (5, 3)), lw=2.0, alpha=0.8)
        for f in nov_km:
            ax.axvline(f, color=COR_NOVA, ls='-', lw=2.2, alpha=0.6)
    for f in rem_km:
        ax_d0.annotate('removida', (f, 1.0), xycoords=('data', 'axes fraction'), ha='center', va='bottom',
                       fontsize=7, color=COR_REMOVIDA)
    for f in nov_km:
        ax_d0.annotate('nova', (f, 1.0), xycoords=('data', 'axes fraction'), ha='center', va='bottom',
                       fontsize=7, color=COR_NOVA)

    handles = [
        Line2D([0], [0], color='#555555', lw=1.0, ls='--', label='Funcional (IGG l2)'),
        Line2D([0], [0], color='#1F4E79', lw=1.2, ls=':',  label='Estrutural (d0 PELT normal)'),
        Line2D([0], [0], color='#C65A57', lw=1.1, ls='-',  label='Combinada (2 etapas)'),
        Line2D([0], [0], color=COR_REMOVIDA, lw=2.0, ls=(0, (5, 3)), label='quebra removida neste passo'),
        Line2D([0], [0], color=COR_NOVA, lw=2.2, ls='-', alpha=0.6, label='quebra nova neste passo'),
    ]
    ax_d0.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 1.07), ncol=5, fontsize=8, frameon=False)
    pen = p['beta'] if alvo == 'estrutural' else p['alpha']
    ref = '  =  REFERENCIA (v8)' if i == i_ref else ''
    ax_d0.set_title(f"SRE {sre} | passo {i + 1}/{n}: {nome_pen} = {pen:.3g} ({p['fator']:.2f}x da referencia){ref}\n"
                    f"Func={len(p['func_km']) + 1} | Estrut={len(p['estrut_km']) + 1} | Comb={len(p['comb_km']) + 1} segmentos",
                    pad=34)

    # (3a) caminho CROPS: onde este passo esta na curva beta x quebras (B&K Fig. 9)
    cam = p['cam']
    if cam is not None and len(cam['Ks']) > 1:
        b_max = max(cam['sup'][1] * 1.15, beta_min + 1, (p['beta'] or 0) * 1.1)
        g = np.linspace(beta_min, b_max, 400)
        ax_cr.step(g, k_otimo(cam['Ks'], cam['Q'], g), where='post', color='#1F4E79', lw=1.2)
        ax_cr.plot(cam['inf'][1:], cam['Ks'][1:], 'o', color='#1F4E79', ms=3)
        K = len(p['estrut_s'])
        if K in cam['Ks']:
            j = int(np.flatnonzero(cam['Ks'] == K)[0])
            ax_cr.axvspan(cam['inf'][j], min(cam['sup'][j], b_max), color='#C65A57', alpha=0.12,
                          label=f'K={K} e otimo neste intervalo')
        if prev is not None and prev['beta'] is not None:
            ax_cr.axvline(prev['beta'], color=COR_REMOVIDA, ls='--', lw=1.0, label=f"passo anterior: {prev['beta']:.2f}")
        ax_cr.axvline(p['beta'], color='#C65A57', lw=1.6, label=f"este passo: beta = {p['beta']:.2f}")
        ax_cr.legend(fontsize=7.5)
    else:
        ax_cr.text(0.5, 0.5, 'SRE sem d0: sem caminho CROPS', ha='center', va='center', transform=ax_cr.transAxes)
    ax_cr.set_xlabel('penalidade beta'); ax_cr.set_ylabel('n. de quebras (K)')
    ax_cr.set_title('Caminho de solucoes (CROPS): quebras x beta', fontsize=9); ax_cr.grid(True, ls=':', alpha=0.5)

    # (3b) compromisso ajuste x complexidade: a reta de inclinacao -beta toca a envoltoria na solucao escolhida
    if cam is not None and len(cam['Ks']) > 1 and len(p['estrut_s']) in cam['Ks']:
        Ks, Q = cam['Ks'], cam['Q']
        K = len(p['estrut_s']); Qk = float(Q[Ks == K][0])
        ax_tr.plot(Ks, Q, 'o-', color='#1F4E79', ms=4, lw=1.0, label='solucoes otimas (envoltoria)')
        xs = np.array([Ks.min(), Ks.max()])
        ax_tr.plot(xs, Qk - p['beta'] * (xs - K), color='#C65A57', lw=1.2, ls='--',
                   label=f'Q + beta.K constante (inclinacao -{p["beta"]:.2f})')
        if prev is not None and len(prev['estrut_s']) in Ks:
            K0 = len(prev['estrut_s'])
            ax_tr.plot([K0], [float(Q[Ks == K0][0])], 'o', color=COR_REMOVIDA, ms=9, label='passo anterior')
        ax_tr.plot([K], [Qk], 'o', color='#C65A57', ms=10, label=f'este passo: K={K}')
        pad = 0.08 * np.ptp(Q) if np.ptp(Q) > 0 else 1.0
        ax_tr.set_ylim(Q.min() - pad, Q.max() + pad)
        ax_tr.legend(fontsize=7.5)
    ax_tr.set_xlabel('n. de quebras K'); ax_tr.set_ylabel('custo Q (-2 log-verossimilhanca)')
    ax_tr.set_title('Ajuste x complexidade: a penalidade escolhe o ponto da envoltoria', fontsize=9)
    ax_tr.grid(True, ls=':', alpha=0.5)

    # (4) explicacao em texto
    ax_tx.text(0.0, 1.0, '\n'.join(explicar_passo(a, p, prev, cam, alvo, nome_pen)), ha='left', va='top',
               fontsize=8.5, family='monospace', transform=ax_tx.transAxes, wrap=True)

    png_path = os.path.join(output_dir, f'{sre}_{i + 1:02d}_{nome_pen}_{pen:.3g}.png')
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight')
    return png_path

def plot_sre_sens(sre, a, ps, i_ref, cores, output_dir, dpi, beta_min, alvo, nome_pen, estavel=None):
    """Resumo da sequencia: posicao das quebras em cada passo (uma linha por penalidade)."""
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg

    n = len(ps)
    r = max(1.0, n / 8)
    p0 = ps[i_ref]
    km_c = a['centro']; d0 = a['d0']; igg = a['igg']; iri = a['iri']
    func_km = p0['func_km']; estrut_km = p0['estrut_km']; comb_km = p0['comb_km']
    pens = [p['beta'] if alvo == 'estrutural' else p['alpha'] for p in ps]

    fig = Figure(figsize=(14, 9.5 + 1.5 * r))
    FigureCanvasAgg(fig)
    gs = fig.add_gridspec(5, 2, height_ratios=[1.15, 0.75 * r, 0.75 * r, 0.85, 1.0], hspace=0.40, wspace=0.22)
    ax_d0  = fig.add_subplot(gs[0, :])
    ax_rv  = fig.add_subplot(gs[1, :], sharex=ax_d0)
    ax_rc  = fig.add_subplot(gs[2, :], sharex=ax_d0)
    ax_igg = fig.add_subplot(gs[3, :], sharex=ax_d0)
    ax_elb = fig.add_subplot(gs[4, 0])
    ax_n   = fig.add_subplot(gs[4, 1])

    ax_d0.plot(km_c, d0, color='#333333', lw=0.8, alpha=0.75)
    lim = [0] + p0['estrut_s'] + [len(d0)]
    for i in range(len(lim) - 1):
        lo, hi = lim[i], lim[i + 1]
        seg = d0[lo:hi]
        if len(seg) and np.isfinite(seg).all():
            m = float(seg.mean()); sd = float(seg.std(ddof=1)) if len(seg) > 1 else 0.0
            x0, x1 = _km_lim(a, lo, hi)
            ax_d0.hlines(m, x0, x1, color='#C65A57', lw=2.0)
            ax_d0.fill_between([x0, x1], m - sd, m + sd, color='#C65A57', alpha=0.15)
    ax_d0.set_ylabel('d0 (um)')

    chave = 'estrut_km' if alvo == 'estrutural' else 'func_km'
    for ax, k, rot in ((ax_rv, chave, f'quebras {alvo}'), (ax_rc, 'comb_km', 'quebras combinada')):
        for i, p in enumerate(ps):
            ax.vlines(p[k], i - 0.38, i + 0.38, color=cores[i], lw=2.2 if i == i_ref else 1.4)
        ax.set_yticks(range(n))
        ax.set_yticklabels([f"{nome_pen} {pv:.3g} ({len(p[k]) + 1} seg)" for pv, p in zip(pens, ps)],
                           fontsize=7.5 if n <= 12 else 6.5)
        ax.get_yticklabels()[i_ref].set_fontweight('bold')
        ax.set_ylim(-0.6, n - 0.4)
        ax.set_ylabel(rot + '\n(penalidade crescente para cima)', fontsize=8.5)
        ax.grid(True, axis='x', ls=':', alpha=0.5)

    ax_igg.plot(km_c, igg, color='#C65A57', lw=0.95)
    ax_igg.set_ylabel('IGG', color='#C65A57'); ax_igg.set_xlabel('Distancia Linear (km)')
    ax_iri = ax_igg.twinx()
    ax_iri.plot(km_c, iri, color='#2E7D32', lw=0.8, alpha=0.65)
    ax_iri.set_ylabel('IRI (m/km)', color='#2E7D32')

    for ax in (ax_d0, ax_igg):
        ax.grid(True, linestyle=':', alpha=0.5)
        for f in func_km:
            ax.axvline(f, color='#555555', ls='--', lw=0.9, alpha=0.8)
        for f in estrut_km:
            ax.axvline(f, color='#1F4E79', ls=':', lw=1.2, alpha=0.85)
        for f in comb_km:
            ax.axvline(f, color='#C65A57', ls='-', lw=1.1, alpha=0.75)

    handles = [
        Line2D([0], [0], color='#555555', lw=1.0, ls='--', label='Funcional (IGG l2)'),
        Line2D([0], [0], color='#1F4E79', lw=1.2, ls=':',  label='Estrutural (d0 PELT normal)'),
        Line2D([0], [0], color='#C65A57', lw=1.1, ls='-',  label='Combinada (2 etapas)'),
    ]
    ax_d0.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=3,
                 fontsize=8, frameon=False)   # fora da area do grafico
    ax_d0.set_title(f"SRE {sre} | resumo de {n} passos de {nome_pen} | referencia ({nome_pen}={pens[i_ref]:.3g}): "
                    f"Func={len(func_km)+1} | Estrut={len(estrut_km)+1} | Comb={len(comb_km)+1}", pad=24)

    cam = p0['cam']
    if cam is not None and len(cam['Ks']) > 1:
        b_max = max([cam['sup'][1] * 1.1, beta_min + 1] + [p['beta'] * 1.05 for p in ps if p['beta'] is not None])
        g = np.linspace(beta_min, b_max, 300)
        ax_elb.step(g, k_otimo(cam['Ks'], cam['Q'], g), where='post', color='#1F4E79', lw=1.2)
        ax_elb.plot(cam['inf'][1:], cam['Ks'][1:], 'o', color='#1F4E79', ms=3)
        vistos = set()
        for i, p in enumerate(ps):
            if p['beta'] is not None and round(p['beta'], 6) not in vistos:
                vistos.add(round(p['beta'], 6))
                ax_elb.axvline(p['beta'], color=cores[i], lw=1.8 if i == i_ref else 1.0)
        if estavel is not None:
            b_sup = estavel[1] if estavel[1] is not None else b_max
            ax_elb.axvspan(estavel[0], b_sup, color='#C65A57', alpha=0.10,
                           label=f"K={estavel[2]} otimo: beta {estavel[0]:.1f} a {b_sup:.1f}")
        if p0['cotovelo'] is not None:
            ax_elb.axvline(p0['cotovelo'][1], color='#555555', ls='--', lw=1.0,
                           label=f"cotovelo SRE: {p0['cotovelo'][1]:.2f} (K={p0['cotovelo'][0]})")
        if estavel is not None or p0['cotovelo'] is not None:
            ax_elb.legend(fontsize=7.5)
    ax_elb.set_xlabel('penalidade beta'); ax_elb.set_ylabel('n. de quebras')
    ax_elb.set_title('Diagnostico CROPS com os beta testados (B&K Fig. 9)', fontsize=9)
    ax_elb.grid(True, ls=':', alpha=0.5)

    for k, cor, ls, rot in [('func_km', '#555555', '--', 'funcional'), ('estrut_km', '#1F4E79', ':', 'estrutural'),
                            ('comb_km', '#C65A57', '-', 'combinada')]:
        ax_n.plot(pens, [len(p[k]) + 1 for p in ps], 'o', ls=ls, color=cor, lw=1.3, ms=4, label=rot)
    ax_n.axvline(pens[i_ref], color='#C65A57', lw=1.0, alpha=0.6)
    ax_n.set_xlabel(f'penalidade {nome_pen}'); ax_n.set_ylabel('n. de segmentos')
    tit_n = 'Segmentos por passo'
    if estavel is not None and ps[i_ref]['beta']:
        b0 = ps[i_ref]['beta']
        tit_n += (f" | estrutural estavel de {estavel[0] / b0:.2f}x a "
                  + (f"{estavel[1] / b0:.2f}x" if estavel[1] is not None else 'inf') + " da referencia")
    ax_n.set_title(tit_n, fontsize=9); ax_n.legend(fontsize=7.5); ax_n.grid(True, ls=':', alpha=0.5)

    png_path = os.path.join(output_dir, f'{sre}_00_resumo.png')
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight')
    return png_path

def sequencia_sre(sre, a, alpha_ref, beta_ref, origem, cam, cotovelo, estavel, output_dir, dpi, beta_min, graficos):
    """Pasta do SRE: resumo + um grafico explicativo por penalidade crescente; devolve (pngs, tabela dos passos)."""
    import matplotlib
    ps = passos_sre(sre, a, alpha_ref, beta_ref, origem, cam, cotovelo)
    n = len(ps)
    i_ref = int(np.argmin([abs(np.log(p['fator'])) for p in ps]))
    tab = []
    for i, p in enumerate(ps):
        prev = ps[i - 1] if i else None
        chave = 'estrut_s' if SENS_ALVO == 'estrutural' else 'func_s'
        K = len(p['estrut_s'])
        j = int(np.flatnonzero(cam['Ks'] == K)[0]) if cam is not None and K in cam['Ks'] else None
        tab.append({'sre': sre, 'passo': i + 1, 'referencia': i == i_ref, 'fator': round(p['fator'], 4),
                    'alpha_km': round(p['alpha'], 6), 'beta': round(p['beta'], 4) if p['beta'] is not None else None,
                    'seg_func': len(p['func_s']) + 1, 'seg_estrut': K + 1, 'seg_comb': len(p['comb_s']) + 1,
                    'custo_Q_estrut': round(float(cam['Q'][j]), 3) if j is not None else None,
                    'beta_otimo_de': round(float(cam['inf'][j]), 4) if j is not None else None,
                    'beta_otimo_ate': (round(float(cam['sup'][j]), 4) if j is not None and np.isfinite(cam['sup'][j]) else None),
                    'quebras_removidas_km': ', '.join(f"{float(a['ini'][b]):.1f}" for b in sorted(set(prev[chave]) - set(p[chave]))) if prev else '',
                    'quebras_novas_km': ', '.join(f"{float(a['ini'][b]):.1f}" for b in sorted(set(p[chave]) - set(prev[chave]))) if prev else '',
                    'quebras_estrut_km': ', '.join(f'{k:.1f}' for k in p['estrut_km']),
                    'quebras_comb_km': ', '.join(f'{k:.1f}' for k in p['comb_km']),
                    'shapiro_p': p['aud'].get('shapiro_p'), 'ljung_box_p': p['aud'].get('ljung_box_p')})
    pngs = []
    if graficos:
        pasta = os.path.join(output_dir, sre)
        os.makedirs(pasta, exist_ok=True)
        cores = matplotlib.colormaps['viridis'](np.linspace(0.05, 0.90, n))
        pngs.append(plot_sre_sens(sre, a, ps, i_ref, cores, pasta, dpi, beta_min, SENS_ALVO, NOME_PEN, estavel))
        for i, p in enumerate(ps):
            pngs.append(plot_passo(sre, a, p, ps[i - 1] if i else None, i, n, i_ref, pasta, dpi, beta_min, SENS_ALVO, NOME_PEN))
    return pngs, tab

t0 = time.time()
sres_plot = [s for s in SRES_SENS if len(grupos[s]['ini']) >= N_MIN_GRAFICO
             and (SENS_ALVO != 'estrutural' or s in CAMINHOS)]   # sem d0 nao ha sequencia estrutural
_seq = Parallel(n_jobs=min(N_JOBS, max(1, len(sres_plot))), batch_size=2)(
    delayed(sequencia_sre)(s, grupos[s], ALPHA_KM_FUNC_OPT, BETA_SRE.get(s, BETA_REDE), ORIGEM_BETA.get(s, 'rede'),
                           CAMINHOS.get(s), COTOVELO.get(s), ESTAVEL.get(s), OUTPUT_DIR, DPI, BETA_MIN_EF, GERAR_GRAFICOS)
    for s in sres_plot)
arquivos_png = [p for pngs, _ in _seq for p in pngs]
TAB_PASSOS = pd.DataFrame([r for _, tab in _seq for r in tab])
del _seq
for p in (CALIB_PNG, SENS_PNG, ESTAB_PNG):
    if os.path.exists(p):
        arquivos_png.append(p)
TEMPOS['graficos'] = time.time() - t0
print(f"Sequencias: {len(TAB_PASSOS)} passos em {len(sres_plot)} SRE(s) | {len(arquivos_png)} graficos em {TEMPOS['graficos']:.1f}s "
      f"({len(SRES_SENS) - len(sres_plot)} SREs com < {N_MIN_GRAFICO} intervalos ou sem d0 omitidos)")
if len(sres_plot) <= 3:
    print(TAB_PASSOS[['sre', 'passo', 'referencia', 'beta', 'fator', 'seg_estrut', 'seg_comb',
                      'quebras_removidas_km', 'quebras_novas_km']].to_string(index=False))


In [ ]:
# ─── 10. Exportacao da Planilha e ZIP dos Graficos ────────────────────────────
t0 = time.time()
params_df = pd.DataFrame([
    {'parametro': 'INPUT_FILE', 'valor': INPUT_FILE},
    {'parametro': 'SENS_ALVO', 'valor': SENS_ALVO},
    {'parametro': 'SRES_ANALISE', 'valor': 'toda a malha' if SRES_ANALISE is None else ', '.join(SRES_ANALISE)},
    {'parametro': 'SENS_GRADE / SENS_MAX_PASSOS', 'valor': f'{SENS_GRADE} / {SENS_MAX_PASSOS}'},
    {'parametro': 'SENS_FATORES', 'valor': ', '.join(f'{f:g}' for f in FATORES)},
    {'parametro': f'{NOME_PEN} de referencia (fator 1)', 'valor': round(PEN_REF, 6)},
    {'parametro': 'SENS_TOL_KM', 'valor': SENS_TOL_KM},
    {'parametro': 'MODEL_FUNC', 'valor': MODEL_FUNC},
    {'parametro': 'MODEL_ESTRUT', 'valor': f"{MODEL_ESTRUT} sobre d0 bruto (Biswas & Kuna, 2022)"},
    {'parametro': 'MODO_PENALIDADE', 'valor': MODO_PENALIDADE},
    {'parametro': 'BETA_REDE', 'valor': round(BETA_REDE, 4)},
    {'parametro': 'BETA_COTOVELO_REDE', 'valor': round(BETA_COTOVELO_REDE, 4)},
    {'parametro': 'BETA_BIC_3lnN', 'valor': round(BETA_BIC, 4)},
    {'parametro': 'BETA_FIXO', 'valor': BETA_FIXO},
    {'parametro': 'BETA_MIN_EF', 'valor': round(BETA_MIN_EF, 4)},
    {'parametro': 'ALPHA_KM_FUNC_OPT', 'valor': round(ALPHA_KM_FUNC_OPT, 6)},
    {'parametro': 'MARGEM_PEN / N_CAND_PEN', 'valor': f'{MARGEM_PEN} / {N_CAND_PEN}'},
    {'parametro': 'MIN_SIZE / JUMP', 'valor': f'{MIN_SIZE} / {JUMP}'},
    {'parametro': 'VAR_FLOOR_FRAC', 'valor': VAR_FLOOR_FRAC},
    {'parametro': 'FUNDIR_FUNC / ESTRUT / COMB', 'valor': f'{FUNDIR_FUNC} / {FUNDIR_ESTRUT} / {FUNDIR_COMB}'},
    {'parametro': 'ALPHA_TESTE', 'valor': ALPHA_TESTE},
    {'parametro': 'BUFFER_KM / MIN_COMP_COMB_KM', 'valor': f'{BUFFER_KM} / {MIN_COMP_COMB_KM}'},
])

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as xw:
    SENS_REDE.to_excel(xw, sheet_name='SENSIBILIDADE_REDE', index=False)
    TAB_PASSOS.to_excel(xw, sheet_name='PASSOS_SRE', index=False)
    SENS_POR_SRE.to_excel(xw, sheet_name='SENSIBILIDADE_POR_SRE', index=False)
    df_info_s.to_excel(xw, sheet_name='CENARIOS_SRE', index=False)
    for tipo in TIPOS:
        sub = df_seg_s[df_seg_s['tipo'] == tipo]
        if not sub.empty:
            sub.drop(columns=['tipo']).to_excel(xw, sheet_name=tipo.upper(), index=False)
    df_aud_s.to_excel(xw, sheet_name='AUDITORIA', index=False)
    DIAG_REDE.to_excel(xw, sheet_name='DIAGNOSTICO_REDE', index=False)
    if CURVA_FUNC is not None:
        CURVA_FUNC.to_excel(xw, sheet_name='CALIBRACAO_FUNC', index=False)
    params_df.to_excel(xw, sheet_name='PARAMETROS', index=False)

with zipfile.ZipFile(ZIP_FILE, 'w', zipfile.ZIP_STORED) as zf:   # PNGs ja sao comprimidos
    for p in arquivos_png:
        zf.write(p, arcname=os.path.join('graficos', os.path.relpath(p, OUTPUT_DIR)))
    if os.path.exists(OUTPUT_FILE):
        zf.write(OUTPUT_FILE, arcname=os.path.basename(OUTPUT_FILE))
TEMPOS['exportacao'] = time.time() - t0

print(f"OK Planilha: {OUTPUT_FILE}")
print(f"OK ZIP gerado: {ZIP_FILE} ({len(arquivos_png)} graficos + planilha) em {TEMPOS['exportacao']:.1f}s")
print("Tempos (s):", {k: round(v, 1) for k, v in TEMPOS.items()}, f"| total={sum(TEMPOS.values()):.1f}s")

if IN_COLAB:
    files.download(ZIP_FILE)
else:
    print(f"Salvo em: {os.path.abspath(ZIP_FILE)}")
